# **EDA del PI — Grupo 8 (e-commerce REES46)**

> ### Maestria en Ciencia de Datos y Analítica
> ### Proyecto Integrador (EAFIT) — EDA oficial
> #### Sara Martínez Rendon, Yeison Londoño, Heider Zapata, Kelly Enriquez

> ⚠️ **EDA oficial del PI.** Origen de datos pendiente (§2.3.1 paso 4): hoy lee una MUESTRA local por usuario (solo Oct 2019) via `config.py` + `src/`, que **no** estan versionados → el notebook **aun no corre tal cual en el repo**. Pendiente re-fuentearlo a **Silver/Gold** en Databricks tras validar y congelar la Gold (ver el `# TODO` en *Carga de Datos*).


## **Contexto del problema**

Dataset: eCommerce Behavior Data (REES46, Oct 2019)

Las plataformas de e-commerce enfrentan un reto estructural: la gran mayoría de 
los usuarios que visitan un producto nunca llegan a comprarlo. Las tasas de 
conversión promedio de la industria oscilan entre el 1% y el 3%, lo que significa 
que por cada 100 visitas, 97 se van sin generar valor.

Este notebook documenta el proceso completo de análisis: desde la exploración 
inicial de los datos hasta la construcción de un argumento visual aclaratorio 
orientado a la toma de decisiones de negocio.

## **Pregunta de Negocio**

> **¿Qué patrones de comportamiento en el e-commerce revelan oportunidades para intervenir con incentivos y aumentar la conversión?**

Entender dónde, cuándo y por qué los usuarios no completan una compra es el primer paso para identificar a quién vale la pena incentivar. Esa pregunta paraguas se descompone en **tres preguntas de negocio operativas** que guían el análisis y, después, el dashboard:

| # | Pregunta de negocio | Se responde en |
|---|---------------------|----------------|
| **PN1** | **¿Dónde perdemos conversión y cuánto vale recuperarla?** | Funnel por unidad (4.1), abandono de carrito (4.5), premio en \$ (4.8a), marca (4.8c) |
| **PN2** | **¿Quiénes son los clientes valiosos y cómo retenerlos?** | Recurrencia por ocasión (4.6), timing de recompra (4.8b) |
| **PN3** | **¿Cuándo y con qué activar el incentivo?** | Patrón horario e intensidad (4.3), velocidad de decisión (4.7), precio-no-es-freno (4.2) |

## **Hallazgo central**

> **El negocio deja dinero sobre la mesa en electrónica.** Hay dos momentos para capturarlo, ambos concentrados en la misma categoría:

- **(A) Antes de comprar — recuperar carritos abandonados.** El abandono real es del **32,4%** y se concentra en electrónica/electrodomésticos (electronics = 67% del abandono). Son **~\$2,08 M** en juego solo en electronics (82% del premio total de \$2,53 M), con Apple y Samsung al frente. El comprador con intención decide en **minutos** (mediana 2,2 min) → incentivo **inmediato/en pantalla**, en la **franja matutina** (la mañana convierte ~2× por visita).
- **(B) Después de comprar — retener al núcleo recurrente.** El **31,7%** de los compradores son recurrentes y concentran el **69% del revenue** (ticket ~5× mayor: \$1.411 vs \$293). La 2ª compra llega en mediana **1,8 días** y el **85,5%** es en la misma categoría → **nudge de recompra a 24–72 h** en su categoría.

> Una historia, dos palancas (conversión + retención), una sola categoría protagonista: **electrónica**.

## **Apéndice histórico — Generación de la muestra (del taller, YA NO se usa)**

> ⚠️ **Esta sección es solo registro histórico.** El EDA **oficial del PI** ya **no**
> lee la muestra local del taller de Visualización (`muestra_usuarios.parquet`, solo
> Oct 2019). A partir de **«Carga de Datos»** este notebook lee las capas **Silver/Gold v1**
> ya materializadas en Databricks (re-fuente §2.3.1 paso 4). Las celdas siguientes se
> conservan **comentadas** para documentar la metodología de muestreo previa y por qué el
> muestreo uniforme por evento sesgaba los hallazgos de nivel sesión/usuario; **no se ejecutan**.

In [ ]:
#pip install duckdb

In [ ]:
# import duckdb
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:
# Instalar la librería gdown si es necesario
# !pip install --upgrade gdown

# import gdown
# import os

# # 2. Configuración del ID del archivo y la URL de descarga
# FILE_ID = '1kDasOXgXimvPn2Shu3wgZymbqj4_8pDc'  
# url = f'https://drive.google.com/uc?id={FILE_ID}'
# output_file = '2019-Oct.csv'

# # 3. Descargar el archivo solo si no existe localmente (para evitar descargas repetidas)
# if not os.path.exists(output_file):
#     print("Iniciando la descarga del dataset de 5GB desde Google Drive...")
#     # gdown se encarga de gestionar la advertencia de tamaño de Google
#     gdown.download(url, output_file, quiet=False)
#     print("¡Descarga completada!")
# else:
#     print("El archivo ya existe en el entorno local.")

**[Proceso original — descartado]** El código a continuación realizaba un muestreo
**uniforme por evento** (`USING SAMPLE 500000`) sobre `2019-Oct.csv`, buscando 500.000
filas. Es el muestreo que se reemplazó por el muestreo por usuario (ver nota arriba). Se
conserva comentado solo como referencia.


In [ ]:
# con = duckdb.connect()

# # Query para traer:
# # 1. El 100% de los eventos 'purchase' y 'cart'
# # 2. Solo un 2% de los eventos 'view' para no saturar la memoria
# query = """
#     SELECT * FROM read_csv_auto('2019-Oct.csv') 
#     USING SAMPLE 500000;
# """

# # Ejecutamos y guardamos en un DataFrame de Pandas
# df_muestra = con.execute(query).df()

# print(f"Tamaño de la muestra: {len(df_muestra)} filas")
# print(df_muestra['event_type'].value_counts())

**[Resultado del muestreo original — descartado]** Salida del muestreo uniforme por evento
(500.000 registros de `2019-Oct.csv`):

- Tamaño de la muestra: 500.000 filas
- `view` 480.542 · `cart` 10.851 · `purchase` 8.607

> Nota: estas cifras NO corresponden a la muestra de trabajo actual. La muestra vigente
> (por usuario) se describe en la nota metodológica de arriba y se carga en la sección
> **Carga de Datos**.


In [ ]:
#df_muestra.to_csv('muestra_eventos.csv', index=False)

## **Carga de Datos — re-fuente a Silver / Gold v1 (Databricks)**

Este EDA es el **entregable oficial del PI** y lee de las **capas Delta materializadas**
del pipeline Medallion, **no** de la muestra del taller. (Re-fuente §2.3.1 paso 4; doc 00 §12.3.)

**Fuentes**
- **SILVER** — `…/silver/clickstream_clean` · **nivel evento**, limpia y tipada
  (109.5M eventos, Oct+Nov 2019). Alimenta funnel, precio, patrón temporal, marca y Pareto.
- **GOLD v1** — `…/gold/features_session` · **1 fila = 1 `user_session`**
  (22.99M sesiones, tasa de etiqueta 0.0610). Alimenta el perfilado de features y etiqueta (§4.x Gold).

**Por qué no es un *swap* 1:1 muestra→Gold:** la mayoría de los análisis son de **nivel
evento / producto-en-sesión** (funnel, precio, horario, marca, Pareto) → leen de **Silver**;
solo el perfilado de features/etiqueta es de **nivel sesión** → lee de **Gold**.

**Disciplina de cuota (doc 02 §4).** Silver **no se re-escanea por gráfico**: toda la
agregación pesada vive en la **«Capa de agregados»** de más abajo, que escanea Silver un
número **acotado** de veces, calcula en Spark y baja a pandas **tablas pequeñas**. Los
gráficos (secciones 1–4) consumen esos objetos en pandas/Matplotlib, sin tocar Spark.

**Cambios de esquema respecto al crudo/muestra** (Silver ya viene enriquecida):
`category_main` → **`macro_category`** · `day_of_week` → **`day_name`** · `hour`, `date`
ya existen como columnas · **no existe** `category_code` (se separó en
`macro_category`/`sub_category`/`item_type`).

**Alcance temporal y anti-fuga.** El EDA descriptivo (funnel, precio, horario, revenue) se
calcula sobre **Oct+Nov**: es estadística de negocio para la Pregunta de Oro y el tablero,
**no** features de modelo → no hay fuga. La **Gold v1** ya nace **anti-fuga** a nivel feature
(solo comportamiento previo al primer `cart`/`purchase`). ⚠️ Cualquier inferencia para el
**diseño de features** (Sara) debe restringirse a **octubre** (split temporal Oct entrena / Nov prueba).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SETUP — EDA oficial del PI sobre Silver / Gold v1 (Databricks · PySpark)
# ════════════════════════════════════════════════════════════════════════════
# Re-fuente (§2.3.1 paso 4): este notebook YA NO lee la muestra local del taller
# (data/processed/muestra_usuarios.parquet, solo Oct 2019). Lee las capas Delta
# materializadas del Medallion. La agregación pesada vive en la "Capa de agregados"
# (celdas siguientes); aquí solo cargamos las capas y verificamos.

from pyspark.sql import functions as F, Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente para toda la exploración
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# ── Rutas de las capas Delta (Volume de Unity Catalog) ───────────────────────
SILVER_PATH = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
GOLD_PATH   = "/Volumes/workspace/default/e_commerce/gold/features_session"
# Delta TEMPORAL para la tabla UNIT (producto-en-sesión). NO es contrato Gold:
# prefijo "_tmp_", se puede borrar al cerrar el día (ver última celda del notebook).
TMP_UNITS   = "/Volumes/workspace/default/e_commerce/gold/_tmp_eda_units"

silver = spark.read.format("delta").load(SILVER_PATH)
gold   = spark.read.format("delta").load(GOLD_PATH)

# Conteos base (se reutilizan más abajo para no re-escanear)
N_SILVER = silver.count()
N_GOLD   = gold.count()

print(f"SILVER : {N_SILVER:,} eventos  ×  {len(silver.columns)} columnas")
print(f"  columnas: {silver.columns}")
print(f"GOLD   : {N_GOLD:,} sesiones ×  {len(gold.columns)} columnas")
print(f"  columnas: {gold.columns}")

In [ ]:
# Vista previa de Silver (nivel evento)
silver.limit(5).toPandas()

## **Capa de agregados EDA (Spark → pandas)**

**Idea central (cuota):** todo el cómputo distribuido se concentra **aquí**. Esta capa
escanea Silver un número **acotado** de veces, agrega en Spark y baja a pandas **tablas
pequeñas**. Las secciones 1–4 que siguen son **solo pandas/Matplotlib** sobre estos
objetos → **ningún gráfico re-escanea Silver** (doc 02 §4; doc 00 §12.3).

**Qué construye, en orden:**
1. **Tabla `units`** (unidad = producto-en-sesión `user_session × product_id`) →
   **Delta temporal**. Es el grano correcto del funnel (una sesión que agrega A y compra B
   no "abandona A"). Se materializa en Delta porque `.cache()` es inestable en serverless
   (commit `2781320`). Reúsa para 4.1 / 4.2 / 4.5 / 4.7 / 4.8.
2. **Funnel por unidad** (`gf` global, `cat_funnel` por categoría) — porta `src/funnel.py` a Spark.
3. **Agregados event-level** desde Silver: completitud/calidad, precio por producto
   (`prod_price_pd`), histograma y cuantiles de precio, conteos `hora × event_type` (`hora_ev`).
4. **Agregados unit-level** desde `units`: revenue en juego (`prize`), marca en electronics
   (`marca_elec`), velocidad de decisión (`decision_pd`).
5. **`purchases_pd`** — los **1.66M eventos `purchase`** traídos a pandas (caben de sobra):
   alimentan 4.3 / 4.4 / 4.6 / 4.8b / 4.8d casi sin cambios respecto al código original.

> **Tras correr esta capa, valida los `print` de shapes/totales** antes de seguir a las
> secciones de gráficos. Si algún agregado falla, el resto del notebook no se ve afectado
> (cada celda es independiente).

In [ ]:
# ── (1) Tabla UNIT: unidad de análisis = producto-en-sesión ───────────────────
# groupBy(user_session, product_id). Un solo escaneo de Silver; se materializa como
# Delta TEMPORAL para reusarla sin re-escanear (en serverless .cache() es inestable).
# first_view_ts / first_purchase_ts permiten calcular la velocidad de decisión (4.7)
# SIN un escaneo adicional de Silver.
# Idempotente: solo reconstruye si el Delta temporal no existe (o si se fuerza).
# Así re-correr esta celda NO vuelve a escanear los 109.5M eventos (cuida la cuota).
FORCE_REBUILD_UNITS = False
try:
    _units_exist = len(dbutils.fs.ls(TMP_UNITS)) > 0
except Exception:
    _units_exist = False

if FORCE_REBUILD_UNITS or not _units_exist:
    units_build = (silver.groupBy("user_session", "product_id").agg(
        F.max(F.when(F.col("event_type") == "view",     1).otherwise(0)).cast("boolean").alias("has_view"),
        F.max(F.when(F.col("event_type") == "cart",     1).otherwise(0)).cast("boolean").alias("has_cart"),
        F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).cast("boolean").alias("has_purchase"),
        F.expr("percentile_approx(price, 0.5)").alias("price"),          # precio mediano de la unidad
        F.first("macro_category", ignorenulls=True).alias("macro_category"),
        F.first("brand", ignorenulls=True).alias("brand"),
        F.min(F.when(F.col("event_type") == "view",     F.col("event_time"))).alias("first_view_ts"),
        F.min(F.when(F.col("event_type") == "purchase", F.col("event_time"))).alias("first_purchase_ts"),
    ))
    (units_build.write.format("delta").mode("overwrite")
                .option("overwriteSchema", "true").save(TMP_UNITS))
    print("UNIT reconstruida desde Silver.")
else:
    print("UNIT ya existe en el Volume; se reutiliza (FORCE_REBUILD_UNITS=False).")

units = spark.read.format("delta").load(TMP_UNITS)
N_UNITS = units.count()
print(f"UNIT (producto-en-sesión): {N_UNITS:,} unidades → {TMP_UNITS}")
units.limit(5).toPandas()

In [ ]:
# ── (2) Funnel por unidad — porta src/funnel.py a Spark sobre la tabla UNIT ───
# reached_cart = unidades con cart O purchase (algunas compras no traen evento cart
# explícito). Partición por etapa más profunda: view_only / cart_only / purchased.
def global_funnel_spark(u):
    r = u.agg(
        F.count("*").alias("n_units"),
        F.sum(F.when(F.col("has_cart") | F.col("has_purchase"), 1).otherwise(0)).alias("reached_cart"),
        F.sum(F.when(F.col("has_purchase"), 1).otherwise(0)).alias("reached_purchase"),
    ).first()
    n_units, reached_cart, purchased = r["n_units"], r["reached_cart"], r["reached_purchase"]
    return {
        "n_units": n_units, "reached_cart": reached_cart, "reached_purchase": purchased,
        "purchased": purchased,
        "cart_only": reached_cart - purchased,
        "view_only": n_units - reached_cart,
        "cart_rate": reached_cart / n_units * 100,
        "conv_rate": purchased / n_units * 100,
        "cart_to_purchase": (purchased / reached_cart * 100) if reached_cart else 0.0,
    }

def category_funnel_spark(u, min_units=500, cat_col="macro_category", missing="Unknown"):
    # "Unknown" es el placeholder de categoría imputado en Silver (~32% de eventos):
    # se trata como FALTANTE y se excluye del desglose por categoría (igual que los nulos
    # en la versión original sobre la muestra).
    g = (u.filter(F.col(cat_col).isNotNull() & (F.col(cat_col) != missing)).groupBy(cat_col).agg(
            F.count("*").alias("units"),
            F.sum(F.when(F.col("has_cart") | F.col("has_purchase"), 1).otherwise(0)).alias("reached_cart"),
            F.sum(F.when(F.col("has_purchase"), 1).otherwise(0)).alias("purchased"),
        ))
    pdf = g.filter(F.col("units") >= min_units).toPandas().set_index(cat_col)
    pdf["cart_rate"] = pdf["reached_cart"] / pdf["units"] * 100
    pdf["conv_rate"] = pdf["purchased"] / pdf["units"] * 100
    pdf["cart_to_purchase"] = (pdf["purchased"] / pdf["reached_cart"] * 100).where(pdf["reached_cart"] > 0, 0.0)
    return pdf.sort_values("conv_rate", ascending=False)

gf = global_funnel_spark(units)
cat_funnel = category_funnel_spark(units, min_units=500)

print("Funnel global — unidad = producto-en-sesión")
print(f"  Unidades                     : {gf['n_units']:,}")
print(f"  Llegan a carrito             : {gf['reached_cart']:,}  ({gf['cart_rate']:.2f}%)")
print(f"  Compran                      : {gf['reached_purchase']:,}  ({gf['conv_rate']:.2f}%)")
print(f"  Cart -> purchase (cierre)    : {gf['cart_to_purchase']:.1f}%")
print(f"  view_only / cart_only / purchased: {gf['view_only']:,} / {gf['cart_only']:,} / {gf['purchased']:,}")
print(f"\ncat_funnel: {cat_funnel.shape[0]} categorías (>=500 unidades)")
cat_funnel[["units", "reached_cart", "purchased", "cart_rate", "conv_rate", "cart_to_purchase"]].round(2)

In [ ]:
# ── (3) Agregados event-level desde Silver (escaneos acotados → pandas chico) ──
COLS_BIZ = ["event_time", "event_type", "product_id", "category_id", "brand",
            "price", "user_id", "user_session", "macro_category", "sub_category", "item_type"]

# "Unknown" es el placeholder imputado en Silver para brand (~14%) y macro_category (~32%).
# Se cuenta como FALTANTE (no como valor real) para una completitud honesta.
MISSING = "Unknown"

# (3a) Completitud por columna + calidad, en UNA sola pasada.
_agg = [F.count(F.when(F.col(c).isNotNull(), c)).alias(f"nn_{c}") for c in COLS_BIZ]
_agg += [F.count(F.when(F.col("price") <= 0, True)).alias("price_le0"),
         F.count(F.when(F.col("brand").isNull() | (F.col("brand") == MISSING), True)).alias("brand_missing"),
         F.count(F.when(F.col("macro_category").isNull() | (F.col("macro_category") == MISSING), True)).alias("mc_missing")]
_row = silver.agg(*_agg).first()
completitud_pd = pd.Series({c: _row[f"nn_{c}"] / N_SILVER * 100 for c in COLS_BIZ})
# brand y macro_category: "Unknown" se descuenta de la completitud (es faltante imputado).
completitud_pd["brand"]          = (N_SILVER - _row["brand_missing"]) / N_SILVER * 100
completitud_pd["macro_category"] = (N_SILVER - _row["mc_missing"]) / N_SILVER * 100
completitud_pd = completitud_pd.sort_values()
# Duplicados de fila completa (Silver ya viene dedup en el pipeline; lo verificamos).
dups_silver = N_SILVER - silver.dropDuplicates().count()
quality_counts = {
    "category\nUnknown": int(_row["mc_missing"]),   # macro_category imputada (~32%)
    "brand\nUnknown":    int(_row["brand_missing"]),  # brand imputada (~14%)
    "Duplicados":        int(dups_silver),
    "Precio = $0":       int(_row["price_le0"]),
}
event_counts = pd.Series(
    {r["event_type"]: r["count"] for r in silver.groupBy("event_type").count().collect()}
).reindex(["view", "cart", "purchase"])

# (3b) Precio por PRODUCTO único (median por product_id) → pandas (pocos cientos de miles).
prod_price_pd = (silver.groupBy("product_id").agg(
        F.expr("percentile_approx(price, 0.5)").alias("price_prod"),
        F.first("macro_category", ignorenulls=True).alias("macro_category"))
    .toPandas())
# Agregado por categoría (insumo de 4.2): media/mediana de los precios-por-producto.
# Se excluye "Unknown" (categoría imputada) del desglose.
precio_cat = (prod_price_pd[prod_price_pd["macro_category"].notna() & (prod_price_pd["macro_category"] != MISSING)]
              .groupby("macro_category")["price_prod"]
              .agg(precio_promedio="mean", precio_mediana="median", n_productos="count")
              .round(2).sort_values("precio_promedio", ascending=False))

# (3c) Cuantiles de precio por event_type (boxplot 4.2/§1, dibujado con ax.bxp).
price_q = (silver.filter(F.col("price") > 0).groupBy("event_type").agg(
        F.expr("percentile_approx(price, array(0.0,0.25,0.5,0.75,1.0))").alias("q"),
        F.mean("price").alias("mean"),
        F.expr("percentile_approx(price, 0.5)").alias("median"),
        F.stddev("price").alias("std"))
    .toPandas().set_index("event_type"))

# (3d) Histograma de precio a nivel evento (bucketizado en Spark; cap visual a 2000).
PRICE_CAP, NBINS = 2000.0, 60
_bw = PRICE_CAP / NBINS
price_hist = (silver.filter(F.col("price") > 0)
    .select(F.least(F.floor(F.col("price") / F.lit(_bw)), F.lit(NBINS - 1)).cast("int").alias("bin"))
    .groupBy("bin").count().orderBy("bin").toPandas())
price_hist["price_left"] = price_hist["bin"] * _bw   # borde izquierdo del bin (USD)

# (3e) Conteos (hora × event_type) → 24×3 (patrón temporal 4.3).
hora_ev = (silver.groupBy("hour").pivot("event_type", ["view", "cart", "purchase"]).count()
           .orderBy("hour").toPandas().set_index("hour").fillna(0).astype("int64"))
hora_ev.columns = ["views", "carts", "purchases"]

print(f"completitud_pd: {len(completitud_pd)} cols | quality_counts: {quality_counts}")
print(f"prod_price_pd: {prod_price_pd.shape} | precio_cat: {precio_cat.shape} | "
      f"price_q: {price_q.shape} | price_hist: {price_hist.shape} | hora_ev: {hora_ev.shape}")

In [ ]:
# ── (4) Agregados unit-level desde la tabla UNIT (sin re-escanear Silver) ──────
# "Unknown" = placeholder de categoría/marca imputado en Silver → se trata como faltante.
MISSING = "Unknown"

# (4a) Revenue en juego en carritos abandonados (4.8a): unidad con cart y sin purchase.
_ab = units.filter(F.col("has_cart") & (~F.col("has_purchase"))).agg(
        F.count("*").alias("n"), F.sum("price").alias("rev")).first()
n_aband_units   = int(_ab["n"])
rev_aband_total = float(_ab["rev"] or 0.0)
prize = (units.filter(F.col("has_cart") & (~F.col("has_purchase")) &
                      F.col("macro_category").isNotNull() & (F.col("macro_category") != MISSING))
         .groupBy("macro_category").agg(
             F.count("*").alias("carritos"),
             F.sum("price").alias("revenue_en_juego"))
         .toPandas().set_index("macro_category"))
prize["ticket_medio"] = prize["revenue_en_juego"] / prize["carritos"]
prize = prize.sort_values("revenue_en_juego", ascending=False)

# (4b) Marca dentro de electronics (4.8c): carritos = unidades que llegan a carrito.
_elec = units.filter((F.col("macro_category") == "electronics") &
                     (F.col("has_cart") | F.col("has_purchase")))
n_elec_total   = _elec.count()
n_elec_nobrand = _elec.filter(F.col("brand").isNull() | (F.col("brand") == MISSING)).count()
marca_elec = (_elec.filter(F.col("brand").isNotNull() & (F.col("brand") != MISSING)).groupBy("brand").agg(
        F.count("*").alias("carritos"),
        F.sum(F.col("has_purchase").cast("int")).alias("comprados"),
        F.expr("percentile_approx(price, 0.5)").alias("ticket"))
    .toPandas().set_index("brand"))
marca_elec["abandonados"] = marca_elec["carritos"] - marca_elec["comprados"]
marca_elec["abandono_%"]  = (marca_elec["abandonados"] / marca_elec["carritos"] * 100).round(1)
marca_elec = marca_elec[marca_elec["carritos"] >= 100].sort_values("abandonados", ascending=False)

# (4c) Velocidad de decisión (4.7): minutos entre primer view y primera compra de la
# unidad comprada. timestamp->double = segundos epoch; /60 = minutos. Sin view previo
# (first_view_ts nulo) la unidad no entra (no hay "decisión" observable).
n_purch_units = units.filter(F.col("has_purchase")).count()
decision_pd = (units.filter(F.col("has_purchase") & F.col("first_view_ts").isNotNull())
    .select(((F.col("first_purchase_ts").cast("double") - F.col("first_view_ts").cast("double")) / 60.0)
            .alias("decision_min"))
    .toPandas())

print(f"prize: {prize.shape} | n_aband_units={n_aband_units:,} rev_aband_total=${rev_aband_total:,.0f}")
print(f"marca_elec: {marca_elec.shape} | electronics carritos={n_elec_total:,} sin_marca={n_elec_nobrand:,}")
print(f"decision_pd: {decision_pd.shape} | unidades compradas={n_purch_units:,} "
      f"(con view previo={len(decision_pd):,})")

In [ ]:
# ── (5) purchases_pd: los eventos `purchase` traídos a pandas (~1.66M filas) ───
# Caben de sobra en el driver. Alimentan 4.3 / 4.4 / 4.6 / 4.8b / 4.8d casi 1:1 con el
# código original (en pandas). Mapeos de nombre: category_main -> macro_category,
# day_of_week -> day_name (ya en Silver).
purchases_pd = (silver.filter(F.col("event_type") == "purchase")
    .select("event_time", "hour", "day_name", "user_id", "user_session",
            "product_id", "price", "macro_category")
    .toPandas())
purchases_pd["event_time"] = pd.to_datetime(purchases_pd["event_time"])

print(f"purchases_pd: {purchases_pd.shape[0]:,} eventos purchase × {purchases_pd.shape[1]} columnas")
print(f"  rango: {purchases_pd['event_time'].min()} → {purchases_pd['event_time'].max()}")
print(f"  usuarios con compra: {purchases_pd['user_id'].nunique():,} | "
      f"sesiones con compra: {purchases_pd['user_session'].nunique():,}")
purchases_pd.head()

## **1. Inspección y Calidad de los Datos**

Antes de cualquier análisis, verificamos la estructura del dataset, los tipos
de datos, valores nulos y posibles inconsistencias. Este paso es fundamental
para garantizar que los hallazgos del EDA sean confiables.

> **Nota de lectura:** las gráficas de esta sección son **a nivel evento** (cada fila
> del log cuenta una vez). Sirven para calidad — detectar nulos, duplicados y precios
> anómalos —, **no** para leer conversión: como cada compra arrastra muchos `view`
> previos, la composición view/cart/purchase del log **no es un funnel**. El funnel real
> (por unidad: producto dentro de sesión) se construye en la sección **4.1**.


In [ ]:
# ── Estructura y calidad general (desde Silver / capa de agregados) ───────────
print("=" * 60); print("ESQUEMA DE SILVER (nivel evento)"); print("=" * 60)
silver.printSchema()

print("=" * 60); print(f"VOLUMEN: {N_SILVER:,} eventos · Oct+Nov 2019"); print("=" * 60)

print("\nCOMPLETITUD por columna (% presente en Silver):")
print(completitud_pd.round(2).to_string())

print("\nCALIDAD (Silver ya viene limpia del pipeline — §2.3.1 paso 3):")
for k, v in quality_counts.items():
    print(f"  {k.replace(chr(10), ' '):24s}: {v:,}")

print("\nCOMPOSICIÓN del log de eventos:")
for et, n in event_counts.items():
    print(f"  {et:9s}: {n:,}  ({n / N_SILVER * 100:.2f}%)")

print("\nPRECIO por event_type (USD):")
print(price_q[["mean", "median", "std"]].round(2).to_string())

In [ ]:
# ── Panel de calidad (4 vistas) — consume los agregados de la capa, no df_muestra ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Panel de Calidad de Datos\nSilver (nivel evento) — Oct+Nov 2019',
             fontsize=14, fontweight='bold', y=1.01)

# ── (1) Completitud por columna ───────────────────────────────────────────────
colores_comp = ['#d62728' if v < 90 else '#2ca02c' for v in completitud_pd.values]
axes[0, 0].barh(completitud_pd.index, completitud_pd.values, color=colores_comp)
axes[0, 0].set_xlim(0, 110)
axes[0, 0].axvline(100, color='gray', linestyle='--', linewidth=0.8)
axes[0, 0].set_title('Completitud por Columna (%)', fontweight='bold')
axes[0, 0].set_xlabel('% de valores presentes')
for i, v in enumerate(completitud_pd.values):
    axes[0, 0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# ── (2) Composición del LOG de eventos (nivel evento) — NO es el funnel ────────
event_pct = (event_counts / N_SILVER * 100).round(2)
bars = axes[0, 1].barh(event_pct.index, event_pct.values,
                       color=sns.color_palette('colorblind', 3))
axes[0, 1].invert_yaxis()
axes[0, 1].set_xlim(0, 110)
axes[0, 1].set_title('Composición del Log de Eventos (nivel evento)', fontweight='bold')
axes[0, 1].set_xlabel('% del total de eventos (registros)')
axes[0, 1].annotate('No es el funnel: cada fila es 1 evento.\nFunnel por unidad → sección 4.1',
                    xy=(0.97, 0.05), xycoords='axes fraction', ha='right', va='bottom',
                    fontsize=8, style='italic', color='#555555')
for bar, val in zip(bars, event_pct.values):
    axes[0, 1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

# ── (3) Distribución de precios POR PRODUCTO ÚNICO (cap visual a $2.000) ───────
PCAP = 2000
pp = prod_price_pd['price_prod'].dropna()
pp_cap = pp[pp <= PCAP]
axes[1, 0].hist(pp_cap, bins=60, color=sns.color_palette('colorblind')[0],
                edgecolor='white', linewidth=0.4)
axes[1, 0].axvline(pp.median(), color='#d62728', linestyle='--', linewidth=1.5,
                   label=f"Mediana: ${pp.median():.0f}")
axes[1, 0].set_title(f'Distribución de Precios por Producto Único\n'
                     f'({pp.size:,} productos · cola > ${PCAP:,} recortada)', fontweight='bold')
axes[1, 0].set_xlabel('Precio mediano del producto (USD)')
axes[1, 0].set_ylabel('Número de productos')
axes[1, 0].legend(fontsize=9)

# ── (4) Resumen de problemas de calidad (Silver ya limpia → todo en 0) ────────
colores_prob = sns.color_palette('colorblind', len(quality_counts))
barras = axes[1, 1].bar(list(quality_counts.keys()), list(quality_counts.values()),
                        color=colores_prob)
axes[1, 1].set_title('Registros con Problemas de Calidad (en Silver)', fontweight='bold')
axes[1, 1].set_ylabel('Número de registros')
axes[1, 1].set_ylim(0, max(1, max(quality_counts.values())) * 1.2 + 1)
for bar, val in zip(barras, quality_counts.values()):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f'{val:,}', ha='center', fontsize=9, fontweight='bold')
axes[1, 1].annotate('Silver ya viene limpia (§2.3.1 paso 3):\ndedup + price>0 aplicados en el pipeline.\n'
                    'brand/category sin nulos → imputados (ver §4.8c).',
                    xy=(0.5, 0.80), xycoords='axes fraction', ha='center', va='top',
                    fontsize=8, style='italic', color='#555555')

plt.tight_layout()
plt.show()

## **2. Verificación de limpieza (ya promovida a Silver)**

La limpieza (dedup, `price > 0`, tipado) **ya no se hace en el EDA**: se promovió a la
lógica determinista de **Silver** en el pipeline (§2.3.1 paso 3). Esta sección **verifica**
que Silver cumple esas garantías, en lugar de limpiar una muestra.

In [ ]:
# Verificación de las garantías de Silver (no se modifica nada; Silver es inmutable aquí).
print("Verificación de limpieza (Silver — §2.3.1 paso 3):")
print(f"  Filas (eventos)    : {N_SILVER:,}")
print(f"  Duplicados de fila : {quality_counts['Duplicados']:,}   (esperado 0)")
print(f"  Precio <= 0        : {quality_counts['Precio = $0']:,}   (esperado 0)")
assert quality_counts['Duplicados'] == 0, "Silver tiene duplicados inesperados"
assert quality_counts['Precio = $0'] == 0, "Silver tiene precios <= 0 inesperados"
print("OK — Silver cumple dedup y price>0.")

> Silver es la fuente limpia e **inmutable** del EDA: aquí no se eliminan filas (eso ya
> ocurrió en el pipeline). Si una verificación fallara, el problema estaría en Silver, no
> en el EDA. **Pendiente de pasada 2 (§2.3.1 paso 5):** outliers de precio y sesiones-bot
> (sin umbral acordado todavía — doc 00 §13).

## **3. Variables derivadas (ya materializadas en Silver)**

El *feature engineering* temporal/categórico **ya vive en Silver** (no se recalcula en el
EDA): `hour`, `date`, `day_name`, `day_of_week_num`, y la jerarquía de categoría
`macro_category` / `sub_category` / `item_type`. Antes el EDA derivaba en pandas `hour`,
`day_of_week` y `category_main`; ahora se consumen directo de Silver (mapeos:
`category_main` → `macro_category`, `day_of_week` → `day_name`).

In [ ]:
# Las variables derivadas ya están en Silver; se muestran para confirmar.
derivadas = ["event_time", "date", "hour", "day_name", "day_of_week_num",
             "macro_category", "sub_category", "item_type"]
silver.select(*derivadas).limit(5).toPandas()

## **4. FASE I - Análisis Exploratorio de Datos (EDA) y Hallazgos**

### **4.1 Funnel de Conversión (por unidad)**

Construimos el funnel **por unidad = producto dentro de sesión** `(user_session, product_id)`:
cada producto que un usuario tocó en una sesión se clasifica por su **etapa más profunda**
(vista → carrito → compra) y las tasas se **encadenan**. Esto corrige el cálculo anterior,
que dividía conteos de eventos (`purchase_events / view_events`) — un cociente que **no es un
funnel de unidades** (el denominador se infla con multi-vistas y las tasas no encadenan).

La lógica vive en `src/funnel.py` (`global_funnel`, `category_funnel`), compartida con el
dashboard para que ambos reporten exactamente lo mismo.

> **Nota:** Los registros sin `category_main` se excluyen del análisis **por categoría**
> (no del funnel global).


**Gráfico izquierdo — Funnel global (unidades)**

Cada unidad es un par (sesión, producto). Contamos cuántas unidades alcanzan cada etapa
— vista → carrito → compra — y el % respecto al total de unidades. La tasa **encadenada**
carrito→compra responde la pregunta clave: de las unidades que llegan al carrito, ¿cuántas
terminan en compra?

**Gráfico derecho — Cart rate vs. conv rate por categoría**

Para cada categoría con **≥500 unidades** (para que la tasa sea representativa):
- **cart rate** = % de unidades que llegan al carrito
- **conv rate** = % de unidades que compran

La línea punteada marca la mediana de conv rate como referencia.


In [ ]:
# Funnel por unidad: gf (global) y cat_funnel ya vienen de la "Capa de agregados".
print("Funnel global — unidad = producto dentro de sesión")
print(f"  Unidades (producto-sesión)   : {gf['n_units']:,}")
print(f"  Llegan a carrito             : {gf['reached_cart']:,}  ({gf['cart_rate']:.2f}% de unidades)")
print(f"  Compran                      : {gf['reached_purchase']:,}  ({gf['conv_rate']:.2f}% de unidades)")
print(f"  Cart → purchase (encadenado) : {gf['cart_to_purchase']:.1f}%")
print("  Partición por etapa más profunda (suma 100%):")
for k, lbl in [('view_only', 'solo vista'), ('cart_only', 'carrito sin compra'), ('purchased', 'compra')]:
    print(f"    {lbl:20s}: {gf[k]:,}  ({gf[k] / gf['n_units'] * 100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Funnel de Conversión por Unidad (producto dentro de sesión)',
             fontsize=14, fontweight='bold')

# --- Izquierda: funnel global acumulado ---
etapas = ['Vista', 'Carrito', 'Compra']
valores = [gf['n_units'], gf['reached_cart'], gf['reached_purchase']]
bars = axes[0].barh(etapas, valores, color=sns.color_palette('colorblind', 3))
axes[0].invert_yaxis()
axes[0].set_title('Funnel Global (unidades)', fontweight='bold')
axes[0].set_xlabel('Nº de unidades (producto-sesión)')
axes[0].set_xlim(0, gf['n_units'] * 1.18)
for bar, val in zip(bars, valores):
    pct = val / gf['n_units'] * 100
    axes[0].text(bar.get_width() + gf['n_units'] * 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:,}  ({pct:.1f}%)', va='center', fontsize=10)

# --- Derecha: cart_rate vs conv_rate por categoría ---
cat_sorted = cat_funnel.sort_values('conv_rate', ascending=True)
y = range(len(cat_sorted))
width = 0.4
colores_cb = sns.color_palette('colorblind', 2)
axes[1].barh([i + width / 2 for i in y], cat_sorted['cart_rate'],
             height=width, color=colores_cb[0], label='Cart rate (llega a carrito)')
axes[1].barh([i - width / 2 for i in y], cat_sorted['conv_rate'],
             height=width, color=colores_cb[1], label='Conv rate (compra)')
axes[1].set_yticks(list(y))
axes[1].set_yticklabels(cat_sorted.index)
axes[1].set_title('Cart Rate vs. Conv Rate por Categoría (%)\nunidad = producto-en-sesión · ≥500 unidades',
                  fontweight='bold')
axes[1].set_xlabel('% de unidades')
axes[1].axvline(cat_sorted['conv_rate'].median(), color='gray',
                linestyle='--', linewidth=1, label='Mediana conv rate')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nTabla de funnel por categoría (≥500 unidades), ordenada por conv_rate:")
print(cat_funnel[['units', 'reached_cart', 'purchased', 'cart_rate', 'conv_rate', 'cart_to_purchase']]
      .round(2))

**Hallazgo 1 — funnel por unidad (producto dentro de sesión), datos completos Oct+Nov**

> **Nota sobre comparabilidad:** las tasas son **por unidad** (producto dentro de sesión),
> no por evento. Esta es la versión sobre **toda la data** (Silver, 109.5M eventos); los
> números difieren de la muestra del taller.

**Global.** De **69.56M unidades** (producto-sesión), el **4.61%** llega al carrito y el
**2.23%** compra. De las que llegan al carrito, **48.3%** termina comprando (**51.7% abandona**).
Partición por etapa más profunda: **95.39% solo vista**, **2.39% carrito sin compra**, **2.23% compra**.

**El cierre es parejo (~43–51%); lo que separa a las categorías es el VOLUMEN y la intención.**
A diferencia de la muestra (donde apparel parecía cerrar al 100%), con datos completos casi
todas las categorías cierran entre **42% y 51%**. El diferenciador es el tamaño del pool:
el carrito sin compra se concentra en **electronics (824.508 unidades, ~50% del total)**,
appliances (196.760) y computers (70.100) — juntas **~66%** del abandono.

**Dos ejes de lectura por categoría:**
- **Intención + cierre (lo mejor):** **electronics** combina la mayor intención
  (cart_rate **6.94%**), la mejor conversión (**3.51%**) y el mejor cierre (**50.6%**) — y el
  mayor volumen. Es el terreno del incentivo de cierre (recordatorio de carrito, urgencia, financiación).
- **Baja intención aguas arriba:** **apparel** (cart_rate **1.37%**, conv **0.58%**),
  furniture (1.80%) y accessories (1.61%) casi no llegan al carrito; su freno es de *vitrina*
  (imágenes, tallas, comparación), no de cierre.

**Implicación de negocio:** segmentar la intervención. Incentivo de **cierre** donde el pool
de carritos abandonados es grande (**electronics, appliances**); mejoras **upstream**
(vitrina/ficha) donde casi nadie llega al carrito (apparel, furniture).

In [ ]:
# Slopegraph cart_rate → conv_rate por categoría (la caída entre puntos = carritos que NO
# cierran). Los datos vienen de cat_funnel (capa de agregados).
#
# NOTA (Fase II): el protagonista es PROVISIONAL. Ajusta HIGHLIGHT / CATS_SLOPE si el
# mensaje de la Fase II lo pide.
HIGHLIGHT = 'electronics'
CATS_SLOPE = ['electronics', 'computers', 'apparel']  # líder · peor cierre · problema upstream

sg = cat_funnel.loc[[c for c in CATS_SLOPE if c in cat_funnel.index], ['cart_rate', 'conv_rate']]

fig, ax = plt.subplots(figsize=(10, 6), dpi=120)
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)
ax.spines['bottom'].set_color('#E0E0E0')
ax.tick_params(axis='both', which='both', length=0)
ax.set_yticks([])  # etiquetas directas en cada extremo (Data-to-Ink alto)

color_highlight = '#d62728'  # rojo para la oportunidad
color_context = '#B0B0B0'    # gris para el contexto

for cat, row in sg.iterrows():
    es_hero = (cat == HIGHLIGHT)
    color = color_highlight if es_hero else color_context
    lw = 3 if es_hero else 1.5
    ax.plot(['1. Llegada al Carrito', '2. Cierre de Compra'],
            [row['cart_rate'], row['conv_rate']],
            color=color, linewidth=lw, marker='o', markersize=8)
    ax.text(-0.05, row['cart_rate'], f"{cat.capitalize()} ({row['cart_rate']:.2f}%)",
            color=color, fontsize=11, ha='right', va='center',
            fontweight='bold' if es_hero else 'normal')
    ax.text(1.05, row['conv_rate'], f"{row['conv_rate']:.2f}%",
            color=color, fontsize=11, ha='left', va='center',
            fontweight='bold' if es_hero else 'normal')

h = sg.loc[HIGHLIGHT]
gap = h['cart_rate'] - h['conv_rate']
ax.set_title(f"La oportunidad de cierre: '{HIGHLIGHT.capitalize()}' atrae al carrito y pierde "
             f"{gap:.2f} pp al pagar",
             fontsize=15, fontweight='bold', loc='left', pad=30, color='#333333')
ax.annotate("Aquí se pierde la venta.\nEs el mayor pool de carritos abandonados\n→ máxima oportunidad de incentivo de cierre.",
            xy=(0.5, (h['cart_rate'] + h['conv_rate']) / 2), xytext=(0.5, h['conv_rate'] * 0.5),
            color='#d62728', fontsize=10, ha='center',
            arrowprops=dict(arrowstyle="->", color='#d62728', connectionstyle="arc3,rad=-0.2"))

plt.tight_layout()
plt.show()

El objetivo de la gráfica es mostrar a la gerencia **dónde** concentrar el incentivo de cierre.
Bajo el funnel por unidad, ese lugar es **electronics**: aunque es la categoría más sana, su
volumen hace que la fuga carrito→compra equivalga a ~5.000 ventas perdidas — el mayor pool de
carritos recuperables. *(Protagonista provisional; se confirma al definir el mensaje en la Fase II.)*


No todos los problemas de conversión son iguales. En **apparel** el freno es de *vitrina*
(cart 0.52%, pero quien llega al carrito compra el 100%): hay que mejorar la ficha/imágenes, no
dar descuento de cierre. En **electronics** y **computers** el usuario sí arma el carrito pero
una parte se cae al pagar (computers cierra solo 61.3%): ese es el arquetipo del usuario
**"persuadible"**, donde un incentivo de cierre puede mover la aguja.


### **4.2 Precio promedio por categoría**

Antes de concluir sobre la relación entre precio y conversión, calculamos el precio promedio real de cada categoría. Esto nos permite contrastar 
con las tasas de conversión.

In [ ]:
# Precio por producto único agregado por categoría (precio_cat) ya viene de la capa de
# agregados: mediana del precio por product_id, promediada/medianada por macro_category
# ("Unknown" excluido). Así los productos muy vistos no sobre-pesan el promedio.
print("Precio por producto único, agregado por categoría (sin 'Unknown'):")
print(precio_cat)

In [ ]:
# Cruzamos el precio (por producto único) con la conversión del funnel por unidad (4.1).
# cat_funnel ya está filtrado a categorías con >= 500 unidades; el join inner aplica ese
# mismo umbral y deja fuera categorías con muy pocas vistas.
precio_conv = (cat_funnel[['units', 'purchased', 'cart_rate', 'conv_rate']]
               .join(precio_cat[['precio_promedio', 'precio_mediana']], how='inner')
               .sort_values('conv_rate', ascending=False))

print(f"Cruce precio vs. conversión ({len(precio_conv)} categorías con >=500 unidades):")
print(precio_conv[['precio_mediana', 'precio_promedio', 'units', 'cart_rate', 'conv_rate']]
      .sort_values('precio_mediana', ascending=False))

# Visualización: scatter precio mediana vs. tasa de conversión
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(precio_conv['precio_mediana'], precio_conv['conv_rate'],
           color=sns.color_palette('colorblind')[0], s=100, zorder=3)
for idx, row in precio_conv.iterrows():
    ax.annotate(idx, xy=(row['precio_mediana'], row['conv_rate']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)
ax.set_title('Precio Mediana vs. Tasa de Conversión por Categoría', fontweight='bold')
ax.set_xlabel('Precio mediana (USD)')
ax.set_ylabel('Tasa de conversión (%)')
ax.axhline(precio_conv['conv_rate'].median(), color='gray', linestyle='--',
           linewidth=1, label=f"Mediana conversión: {precio_conv['conv_rate'].median():.1f}%")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de precio (nivel evento) + precio por event_type. Ambos vienen de la capa
# de agregados (price_hist bucketizado en Spark, price_q = cuantiles por percentile_approx),
# por lo que no se traen los 109M precios a pandas.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Izquierda: histograma de price (cap visual a $2.000; cola larga recortada) ---
_w = price_hist['price_left'].diff().median()
axes[0].bar(price_hist['price_left'], price_hist['count'], width=_w, align='edge',
            color='#4c78a8', edgecolor='white', linewidth=0.3)
axes[0].set_title('Distribución de price (nivel evento · cap $2.000)')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

# --- Derecha: boxplot por event_type, dibujado con stats precalculadas (ax.bxp) ---
order = ['view', 'cart', 'purchase']
bxp_stats = []
for et in order:
    q = price_q.loc[et, 'q']           # [min, q1, med, q3, max]
    q0, q1, med, q3, q4 = [float(v) for v in q]
    iqr = q3 - q1
    bxp_stats.append({'label': et, 'med': med, 'q1': q1, 'q3': q3,
                      'whislo': max(q0, q1 - 1.5 * iqr), 'whishi': min(q4, q3 + 1.5 * iqr),
                      'fliers': []})
axes[1].bxp(bxp_stats, showfliers=False, patch_artist=True,
            boxprops=dict(facecolor=sns.color_palette('muted')[0], alpha=0.8))
axes[1].set_title('Precio por event_type (sin outliers extremos)')
axes[1].set_ylabel('Precio (USD)')

plt.tight_layout()
plt.show()

print('─── Precio por event_type (USD) ───')
print(price_q[['mean', 'median', 'std']].round(2).to_string())

**Hallazgo 2 — El precio no es el freno (y los más baratos no convierten mejor).**

Cruzando el precio mediano por **producto único** con la conversión del **funnel por unidad**
(13 categorías con ≥500 unidades), no aparece tendencia: el precio no explica la conversión.
La mejor conversión es **electronics (3.51%)** a un precio mediano intermedio (**$146**); en
cambio dos de las **más baratas** —apparel (**$61**, conv 0.58%) y accessories (**$50**, 0.73%)—
convierten **entre las peores**. Si el precio fuera el freno, lo barato debería cerrar mejor:
ocurre lo contrario. La más cara (sport, $298) tampoco lidera (0.93%). El boxplot lo confirma:
los productos comprados (mediana **$174**) no son más baratos que los vistos (**$165**) —incluso
algo más caros.

**Traducción de negocio:** el obstáculo no es *cuánto cuesta*, sino algo no-monetario:
incertidumbre de talla/ajuste en apparel, dificultad para comparar en accessories, o decisiones
que se completan en otro canal.

**Acción:** en las categorías que no cierran (apparel, accessories) los incentivos NO deberían
ser descuentos de precio, sino reducir la fricción no-monetaria (guías de talla, garantía de
devolución, comparadores). En electronics —el "persuadible" del Hallazgo 1, que ya convierte
bien a precio alto— la palanca es cerrar el carrito (cart→purchase), no bajar el precio.

### **4.3 Patrón Temporal de Compras**

Analizamos cómo se distribuyen los eventos de compra a lo largo del día y la semana. El objetivo es identificar franjas horarias y días con mayor 
actividad de compra — ventanas de oportunidad para activar incentivos.

In [ ]:
# Patrón temporal de compras: sobre purchases_pd (los 1.66M eventos purchase en pandas).
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('4.3 Patrón Temporal de Compras', fontsize=14, fontweight='bold')

# --- Izquierda: compras por hora del día ---
compras_hora = purchases_pd.groupby('hour').size()
axes[0].bar(compras_hora.index, compras_hora.values,
            color=sns.color_palette('colorblind')[0], edgecolor='white')
axes[0].set_title('Compras por Hora del Día', fontweight='bold')
axes[0].set_xlabel('Hora (0-23)')
axes[0].set_ylabel('Número de compras')
axes[0].set_xticks(range(0, 24, 2))
axes[0].axvline(compras_hora.idxmax(), color='#d62728', linestyle='--',
                linewidth=1.5, label=f"Pico: {compras_hora.idxmax()}h")
axes[0].legend()

# --- Derecha: compras por día de la semana (day_name de Silver) ---
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
compras_dia = purchases_pd.groupby('day_name').size().reindex(orden_dias)
axes[1].bar(compras_dia.index, compras_dia.values,
            color=sns.color_palette('colorblind')[1], edgecolor='white')
axes[1].set_title('Compras por Día de la Semana', fontweight='bold')
axes[1].set_xlabel('Día')
axes[1].set_ylabel('Número de compras')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(compras_dia.mean(), color='gray', linestyle='--',
                linewidth=1.2, label=f"Promedio: {compras_dia.mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Compras por hora (top 5):")
print(compras_hora.sort_values(ascending=False).head())
print("\nCompras por día:")
print(compras_dia)

In [ ]:
# Intensidad de compra por hora. hora_ev (conteos hora × event_type) viene de la capa de
# agregados. Normalizamos cada tipo a su distribución horaria y calculamos compras/100 vistas.
hora_tab = hora_ev.copy()   # columnas: views, carts, purchases
hora_tab['%_vistas']   = (hora_tab['views']     / hora_tab['views'].sum()     * 100).round(2)
hora_tab['%_carritos'] = (hora_tab['carts']     / hora_tab['carts'].sum()     * 100).round(2)
hora_tab['%_compras']  = (hora_tab['purchases'] / hora_tab['purchases'].sum() * 100).round(2)
hora_tab['compras_x100_vistas'] = (hora_tab['purchases'] / hora_tab['views'] * 100).round(2)

print('Estadísticas por hora del día:')
print(hora_tab)

cb = sns.color_palette('colorblind')
horas = hora_tab.index
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel izq: distribución horaria NORMALIZADA (cada tipo suma 100% sobre las horas)
axes[0].plot(horas, hora_tab['%_vistas'],   color='#b0b0b0', lw=2,   marker='o', ms=3, label='Vistas')
axes[0].plot(horas, hora_tab['%_carritos'], color=cb[1],     lw=2,   marker='o', ms=3, label='Carritos')
axes[0].plot(horas, hora_tab['%_compras'],  color=cb[0],     lw=2.8, marker='o', ms=4, label='Compras')
axes[0].set_title('Distribución horaria normalizada\n(% de los eventos de cada tipo por hora)', fontweight='bold')
axes[0].set_xlabel('Hora (0-23)')
axes[0].set_ylabel('% de los eventos del tipo')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

# Panel der: intensidad de compra (compras por 100 vistas) por hora
peak_h = int(hora_tab['compras_x100_vistas'].idxmax())
axes[1].bar(horas, hora_tab['compras_x100_vistas'], color=cb[0], edgecolor='white')
axes[1].axvline(peak_h, color='#d62728', ls='--', lw=1.5, label=f'Máx: {peak_h}h')
axes[1].set_title('Intensidad de compra por hora\n(compras por 100 vistas — nivel evento)', fontweight='bold')
axes[1].set_xlabel('Hora (0-23)')
axes[1].set_ylabel('Compras por 100 vistas')
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()

plt.tight_layout()
plt.show()

**Hallazgo 3 — Cuándo se navega ≠ cuándo se compra.**

El tráfico (vistas) pica por la **tarde** —máximo a las **16h**—, pero las **compras** se
concentran en la **mañana**: la franja **7h–11h** acumula el grueso, con **pico a las 9h**
(126.616 compras). Las curvas normalizadas van casi en contrafase: a media tarde hay mucha
gente mirando pero comprando proporcionalmente menos.

**Y la mañana no solo compra más: convierte mejor.** La intensidad (compras por 100 vistas,
nivel evento) es máxima a las **9h (2.26)** y se sostiene alta en toda la franja matutina
(≈2.0–2.2 entre 6h–10h), mientras que por la **tarde —cuando hay MÁS tráfico— cae a ~1.1**
(16h: 1.13, 17h: 1.11). Cada visita matutina rinde casi el **doble**: la tarde es navegación
de baja intención ("vitrineo"); la mañana es decisión de compra.

**Por día**, el **fin de semana lidera en volumen de compras**: domingo (353.609) y sábado
(259.472) superan claramente a los días hábiles (≈200–216 mil). El día pesa menos que la hora,
pero el patrón se invirtió respecto a la muestra del taller (que sugería mitad de semana).

**Implicación de negocio:** concentrar incentivos y push en la **franja matutina (≈7h–11h,
pico 9h)**, cuando el usuario llega en modo decisión y cada visita rinde el doble; reforzar el
fin de semana por volumen. Empujar en la tarde compite contra el "solo estoy mirando".

### **4.4 Patrón Temporal por Categoría**

El hallazgo anterior muestra un pico global de compras a las 9h. Sin embargo, 
ese número agrega el comportamiento de todas las categorías en un solo valor — 
y el análisis del funnel ya nos mostró que las categorías se comportan de forma 
muy distinta entre sí.

Exploramos las 5 categorías con mayor volumen de compras para ver si el 
patrón temporal es uniforme o también varía por categoría.

In [ ]:
# Patrón de compras por hora según categoría (top 5 por volumen). Sobre purchases_pd.
top5_cats = ['electronics', 'appliances', 'computers', 'apparel', 'furniture']

purchases_cat = purchases_pd[purchases_pd['macro_category'].isin(top5_cats)].copy()
compras_hora_cat = (purchases_cat.groupby(['macro_category', 'hour']).size()
                    .reset_index(name='compras'))
totales = compras_hora_cat.groupby('macro_category')['compras'].transform('sum')
compras_hora_cat['pct'] = (compras_hora_cat['compras'] / totales * 100).round(2)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('4.4 Patrón de Compras por Hora según Categoría', fontsize=14, fontweight='bold')
axes = axes.flatten()
colores = sns.color_palette('colorblind', 5)

for i, cat in enumerate(top5_cats):
    data = compras_hora_cat[compras_hora_cat['macro_category'] == cat]
    hora_pico = data.loc[data['pct'].idxmax(), 'hour']
    axes[i].bar(data['hour'], data['pct'], color=colores[i], edgecolor='white')
    axes[i].axvline(hora_pico, color='#d62728', linestyle='--', linewidth=1.5, label=f'Pico: {hora_pico}h')
    axes[i].set_title(cat.capitalize(), fontweight='bold')
    axes[i].set_xlabel('Hora (0-23)')
    axes[i].set_ylabel('% de compras')
    axes[i].set_xticks(range(0, 24, 2))
    axes[i].legend(fontsize=9)

axes[5].set_visible(False)
plt.tight_layout()
plt.show()

print("Hora pico de compras por categoría:")
for cat in top5_cats:
    data = compras_hora_cat[compras_hora_cat['macro_category'] == cat]
    hora_pico = data.loc[data['pct'].idxmax(), 'hour']
    pct_pico = data['pct'].max()
    print(f"  {cat:15s} → {hora_pico}h  ({pct_pico:.1f}% de sus compras)")

**Hallazgo 4 — Patrón horario por categoría: casi todo el negocio compra a media mañana.**

Sobre datos completos, **cuatro de las cinco** categorías de mayor volumen pican a las **9h**,
alineadas con el patrón global del Hallazgo 3:

| Categoría   | Hora pico | % de sus compras en el pico | Perfil |
|-------------|-----------|------------------------------|--------|
| Electronics | 9h        | 7.8% | Media mañana — decisión al arranque del día |
| Computers   | 9h        | 8.1% | Media mañana |
| Appliances  | 9h        | 7.8% | Media mañana |
| Furniture   | 9h        | 7.5% | Media mañana |
| Apparel     | 17h       | 7.1% | **Tarde** — perfil distinto al resto |

La excepción es **apparel**, que pica al final de la **tarde (17h)**: un comportamiento de
compra distinto al de electrónica/electrodomésticos, coherente con su naturaleza de consumo
más impulsivo/vespertino.

**Conexión con el Proyecto Integrador:** combinar la identificación del **usuario persuadible**
(lo que estima el clasificador de propensión) con la **hora de mayor receptividad** por categoría
permite diseñar incentivos más precisos. La efectividad real del momento de activación se validaría
con el **A/B test** del cierre del proyecto (no se estima causalidad desde datos observacionales).

### **4.5 Abandono de carrito (por unidad)**

In [ ]:
# Abandono = unidades que llegan al carrito pero NO compran, consistente con el funnel 4.1.
# Reutiliza gf (funnel global) y cat_funnel (por categoría, >=500 unid) de la capa de agregados.
abandono_global = 100 - gf['cart_to_purchase']   # = cart_only / reached_cart
print(f"Unidades que llegan al carrito: {gf['reached_cart']:,}")
print(f"  compran:   {gf['purchased']:,}  (cierre {gf['cart_to_purchase']:.1f}%)")
print(f"  abandonan: {gf['cart_only']:,}  (abandono {abandono_global:.1f}%)")

aband_cat = cat_funnel.assign(
    abandonados=(cat_funnel['reached_cart'] - cat_funnel['purchased']),
    abandono=100 - cat_funnel['cart_to_purchase'],
).sort_values('abandono', ascending=False)
print('\nAbandono de carrito por categoría:')
print(aband_cat[['reached_cart', 'purchased', 'abandonados', 'cart_to_purchase', 'abandono']].round(1))

cb = sns.color_palette('colorblind')
PERSUADIBLE = {'electronics', 'computers'}  # alta intención (Hallazgo 1) que no cierra
colores = [cb[0] if c in PERSUADIBLE else '#b0b0b0' for c in aband_cat.index]

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(aband_cat.index, aband_cat['abandono'], color=colores, edgecolor='white')
ax.invert_yaxis()
ax.set_xlim(0, aband_cat['abandono'].max() * 1.22)
ax.axvline(abandono_global, color='#d62728', ls='--', lw=1.5,
           label=f'Abandono global: {abandono_global:.1f}%')
ax.set_title('Abandono de carrito por categoría\n(% de unidades que llegan al carrito y no compran)',
             fontweight='bold')
ax.set_xlabel('% de abandono (unidad = producto-en-sesión)')
for b, n in zip(barras, aband_cat['abandonados']):
    ax.text(b.get_width() + 0.4, b.get_y() + b.get_height() / 2,
            f'{int(n):,} carritos', va='center', fontsize=8, color='#555')
ax.legend()
plt.tight_layout()
plt.show()

**Hallazgo 5 — El abandono real es ~52%, y su mayor pool está en electrónica.**

De cada producto que llega al carrito (unidad = producto-en-sesión, consistente con el funnel
de 4.1), **el 51.7% no se compra** (1.660.501 de 3.208.745); el otro 48.3% cierra. Con datos
completos la cifra es mayor que la ~32% de la muestra del taller (artefacto del muestreo por evento).

**El abandono no se reparte parejo, pero no es por peor cierre de electrónica.** Por *tasa*,
las categorías que más abandonan son de bajo volumen: country_yard (60.7%), sport (57.7%),
apparel (57.5%). **Electronics tiene de hecho el cierre MÁS alto (abandono 49.4%)** — pero su
**volumen es enorme**: concentra **824.508 carritos abandonados, ~50% del total**, y con
appliances (196.760) y computers (70.100) suman **~66%**. Ahí está la oportunidad: no en la
tasa, sino en el tamaño del pool.

**Traducción de negocio:** existe un segmento grande de **intención declarada que no cierra**,
concentrado por volumen en **electrónica y electrodomésticos** — el "usuario persuadible". No es
un problema de precio (Hallazgo 2): es recuperación de carrito en categorías caras y de alta consideración.

**Acción:** dirigir incentivos de cierre (recordatorio de carrito, urgencia/stock, financiación)
a los carritos abandonados de **electronics y appliances**, idealmente en la franja matutina de
alta intención (Hallazgo 3). Para apparel el incentivo correcto NO es de carrito sino aguas
arriba (guías de talla, garantía de devolución).

### **4.6 Segmentación compradores one-time vs. recurrentes**

La mayoría de métricas de conversión tratan a todos los compradores como
iguales. Sin embargo, dentro del grupo que sí compró existe una diferencia
crítica: quienes compraron una sola vez y quienes volvieron a comprar.

Este análisis segmenta a los compradores en dos grupos y evalúa cuánto
revenue concentra cada uno. El objetivo es identificar si el grupo recurrente
—aunque pequeño— representa una oportunidad de fidelización
desproporcionadamente valiosa.

**Gráfico izquierdo — Composición de compradores**

Contamos cuántos usuarios únicos compraron exactamente una vez versus más de
una vez. La proporción revela el tamaño del grupo fidelizable.

**Gráfico derecho — Concentración de revenue**

Comparamos el revenue total generado por cada grupo. Si el grupo recurrente
concentra una fracción del revenue superior a su peso en número de usuarios,
confirma el fenómeno de "valor desproporcionado del comprador fiel".

In [ ]:
# Recurrencia = volver a comprar en OTRA ocasión (sesión distinta con compra), no llevar
# varios ítems en un mismo pedido. Sobre purchases_pd (todos los eventos purchase).
# OCASIONES = nº de sesiones distintas con compra (NO eventos purchase).
user_buys = purchases_pd.groupby('user_id').agg(
    ocasiones=('user_session', 'nunique'),   # sesiones distintas con compra
    items=('product_id', 'size'),            # eventos purchase (ítems comprados)
    revenue=('price', 'sum'),
)

one_time = int((user_buys['ocasiones'] == 1).sum())
repeat   = int((user_buys['ocasiones'] >= 2).sum())
n_buyers = one_time + repeat
rev_one_time = user_buys.loc[user_buys['ocasiones'] == 1, 'revenue'].sum()
rev_repeat   = user_buys.loc[user_buys['ocasiones'] >= 2, 'revenue'].sum()
rev_total    = user_buys['revenue'].sum()

print(f"Compradores: {n_buyers:,}")
print(f"  One-time (1 ocasión):  {one_time:,} ({one_time/n_buyers*100:.1f}%)")
print(f"  Recurrentes (>=2):     {repeat:,} ({repeat/n_buyers*100:.1f}%)")
print(f"Revenue de recurrentes: {rev_repeat/rev_total*100:.1f}% del total")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segmentación: Compradores One-Time vs. Recurrentes\n(recurrencia = compra en ≥2 sesiones distintas)',
             fontsize=14, fontweight='bold')

colores = sns.color_palette('colorblind', 2)
labels_seg = ['One-time\n(1 ocasión)', 'Recurrentes\n(≥ 2 ocasiones)']

# --- Izquierda: composición de compradores ---
valores_seg = [one_time, repeat]
bars = axes[0].bar(labels_seg, valores_seg, color=colores, edgecolor='white', width=0.5)
axes[0].set_title('Número de Compradores por Segmento', fontweight='bold')
axes[0].set_ylabel('Número de usuarios')
for bar, val in zip(bars, valores_seg):
    pct = val / n_buyers * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(valores_seg) * 0.01,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

# --- Derecha: concentración de revenue ---
valores_rev = [rev_one_time, rev_repeat]
bars2 = axes[1].bar(labels_seg, valores_rev, color=colores, edgecolor='white', width=0.5)
axes[1].set_title('Revenue Generado por Segmento (USD)', fontweight='bold')
axes[1].set_ylabel('Revenue total (USD)')
for bar, val in zip(bars2, valores_rev):
    pct = val / rev_total * 100
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(valores_rev) * 0.01,
                 f'${val:,.0f}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nUsuarios por número de ocasiones de compra:")
print(user_buys['ocasiones'].value_counts().sort_index().rename('usuarios').head(10))
print(f"\nRevenue promedio por usuario:")
print(f"  One-time:    ${rev_one_time / one_time:,.2f}")
print(f"  Recurrente:  ${rev_repeat / repeat:,.2f}")

**Hallazgo 6 — Un tercio largo de los compradores vuelve y genera ~3 de cada 4 dólares.**

Contando ocasiones (sesiones distintas con compra, no ítems) sobre **toda la data (Oct+Nov)**,
**el 36.8% de los compradores (256.959 de 697.470) vuelve a comprar en otra sesión**; el 63.2%
compra una sola vez. Esto confirma, ya sin depender de una muestra, lo que el muestreo por evento
del taller ocultaba (aparentaba ~97% one-time).

Y el segmento recurrente es **desproporcionadamente valioso**: siendo el 36.8% de los compradores
concentra el **73.8% del revenue**. El comprador recurrente gasta en promedio **$1.450 vs $301**
del one-time — casi **5×** más. La distribución tiene cola larga: 132.059 usuarios con 2 ocasiones,
52.050 con 3, 25.196 con 4, y miles con 5+.

**Traducción de negocio:** la base de clientes no es un mar de compradores únicos; tiene un
**núcleo recurrente que sostiene la mayoría de los ingresos**. Retener y reactivar ese núcleo es
una palanca de valor de primer orden, complementaria a convertir al "usuario persuadible"
(Hallazgos 1 y 5).

**Acción:** proteger y hacer crecer el segmento recurrente (fidelización/puntos, beneficios por
recompra) y trabajar el salto crítico de la 1ª a la 2ª ocasión (post-compra, recordatorio) para
mover one-times al núcleo recurrente.

### **4.7 Velocidad de decisión en sesión**

Los hallazgos anteriores identifican *quién* convierte y *cuándo* del día lo
hace. Este análisis cierra el ciclo preguntando *en cuánto tiempo* ocurre la
decisión de compra dentro de la propia sesión.

Medimos, para cada producto comprado, el tiempo entre su primer `view` y su
compra dentro de la misma sesión (unidad = producto-en-sesión, consistente con
el funnel). Esto equivale a preguntar: una vez que un usuario ve el producto
que terminará comprando, ¿cuántos minutos tarda en decidir?

La respuesta es crítica para el diseño de incentivos: si la mayoría de
compras ocurre en los primeros minutos de sesión, el incentivo debe ser
visible e inmediato (banner en producto, descuento en pantalla). Si la
decisión es lenta, los canales diferidos (email de retargeting, notificación
al día siguiente) tienen más sentido.

> **Nota metodológica:** Se mide a nivel producto-en-sesión: solo productos
> con un `view` y un `purchase` en la misma sesión, con la compra posterior al
> primer `view`. Los tiempos negativos (compra antes del primer view) son
> artefactos residuales del muestreo y se descartan —su número se reporta en la
> salida de la celda—. Con la muestra por usuario casi desaparecen; antes, con
> el muestreo por evento, se perdía ~27% de las sesiones por este motivo.

In [ ]:
# Velocidad de decisión por UNIDAD (producto-en-sesión): minutos entre el primer view del
# producto comprado y su compra. decision_pd ya viene de la capa de agregados (calculado
# en Spark desde first_view_ts/first_purchase_ts de la tabla UNIT).
decision_time = decision_pd['decision_min']
n_neg = int((decision_time < 0).sum())
decision_time = decision_time[decision_time >= 0]   # descartar artefactos negativos

print(f"Unidades (producto-en-sesión) vistas y compradas: {len(decision_pd):,}")
print(f"  descartadas por tiempo negativo: {n_neg} ({n_neg/len(decision_pd)*100:.2f}%)")
print(f"  válidas (>= 0 min): {len(decision_time):,}")
print()
print(decision_time.describe().round(1))
print(f"\nDecisiones en < 5 min:  {(decision_time < 5).mean()*100:.1f}%")
print(f"Decisiones en < 10 min: {(decision_time < 10).mean()*100:.1f}%")
print(f"Decisiones en < 30 min: {(decision_time < 30).mean()*100:.1f}%")
print(f"Decisiones en < 60 min: {(decision_time < 60).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Velocidad de Decisión (primer view del producto comprado → compra)',
             fontsize=14, fontweight='bold')

# --- Izquierda: histograma de tiempos (cap a 60 min para legibilidad) ---
dt_cap = decision_time[decision_time <= 60]
axes[0].hist(dt_cap, bins=30, color=sns.color_palette('colorblind')[0],
             edgecolor='white', linewidth=0.5)
axes[0].axvline(decision_time.median(), color='#d62728', linestyle='--', linewidth=1.8,
                label=f'Mediana: {decision_time.median():.1f} min')
axes[0].axvline(decision_time.mean(), color='#ff7f0e', linestyle=':', linewidth=1.8,
                label=f'Media: {decision_time.mean():.1f} min')
axes[0].set_title('Distribución del Tiempo de Decisión\n(unidades con ≤ 60 min)', fontweight='bold')
axes[0].set_xlabel('Minutos desde primer view hasta purchase')
axes[0].set_ylabel('Número de unidades (producto-en-sesión)')
pct_10 = (decision_time < 10).mean() * 100
axes[0].axvspan(0, 10, alpha=0.08, color='#2ca02c', label=f'{pct_10:.0f}% < 10 min')
axes[0].legend(fontsize=9)

# --- Derecha: distribución por segmentos de tiempo ---
bins = [0, 5, 10, 30, 60, decision_time.max() + 1]
labels_bins = ['< 5 min', '5–10 min', '10–30 min', '30–60 min', '> 60 min']
segmentos = pd.cut(decision_time, bins=bins, labels=labels_bins, right=False)
seg_counts = segmentos.value_counts().sort_index()
seg_pct = (seg_counts / seg_counts.sum() * 100).round(1)

colores_seg = sns.color_palette('colorblind', len(labels_bins))
bars = axes[1].bar(seg_counts.index.astype(str), seg_counts.values,
                   color=colores_seg, edgecolor='white')
axes[1].set_title('Unidades por Segmento de Tiempo de Decisión', fontweight='bold')
axes[1].set_xlabel('Segmento de tiempo')
axes[1].set_ylabel('Número de unidades')
axes[1].tick_params(axis='x', rotation=20)
for bar, val, pct in zip(bars, seg_counts.values, seg_pct.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + seg_counts.max() * 0.01,
                 f'{val:,}\n({pct}%)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

**Hallazgo 7 — La decisión es casi inmediata: mediana de 2.2 minutos.**

Midiendo a nivel producto-en-sesión (tiempo entre el primer `view` del producto comprado y su
compra), la decisión es muy rápida: **mediana de 2.2 minutos** (igual que en la muestra: el
hallazgo se sostiene a escala completa), con el **77.2% de las compras en menos de 5 minutos** y
el **90.8% en menos de 10**. Solo ~1.8% tarda más de 30 minutos, lo que descarta al "comprador
deliberativo" como caso típico. La media (5.0 min) supera a la mediana por una cola corta de
decisiones largas, pero lo normal es un par de minutos.

**Limpieza metodológica:** sobre la data completa solo el **0.45% (6.878 de 1.536.271)** de las
unidades vista-y-comprada tiene tiempo negativo (compra antes del primer view, artefacto residual);
se descartan, quedando **1.529.393 unidades** válidas. (En el muestreo por evento del taller se
descartaba ~27% — aquello era daño del muestreo, no del fenómeno.)

**Traducción de negocio:** cuando el usuario ve el producto que terminará comprando, la decisión
ya está prácticamente tomada en los primeros minutos. No hay una ventana de deliberación larga
que un canal diferido pueda capturar para *acelerar* esa compra.

**Acción:** el incentivo para el comprador con intención activa debe ser **inmediato y en pantalla**
(banner de descuento, "últimas unidades", financiación visible en la ficha), no diferido. Los
canales diferidos (email/push) sirven para *otros* objetivos: recuperar carritos abandonados
(Hallazgo 5) y reactivar al núcleo recurrente (Hallazgo 6).

### **4.8 ¿Dónde está el dinero? Dimensionar la oportunidad**

Los hallazgos 1–7 dicen *a quién, dónde, cuándo y con qué* incentivar, pero en **tasas**. Para decidir presupuesto, el gerente necesita el **tamaño del premio en dinero**. Aquí traducimos los hallazgos a revenue: (a) cuánto está en juego en los carritos abandonados y cuánto vale mover un one-time al núcleo recurrente, (b) el *timing* de la recompra, (c) el reparto por marca en las categorías clave, y (d) qué tan concentrado está el revenue (Pareto).

In [ ]:
# (a) REVENUE EN JUEGO en carritos abandonados. prize / n_aband_units / rev_aband_total
# vienen de la capa de agregados (unidades con cart y sin purchase; "Unknown" excluido).
print(f"Carritos abandonados (unidades): {n_aband_units:,}")
print(f"Revenue total en juego: ${rev_aband_total:,.0f}")
print("\nRevenue en juego por categoría:")
print(prize[['carritos', 'revenue_en_juego', 'ticket_medio']].round(0))

# Escenario de recuperación sobre la categoría con más revenue en juego
top_cat = prize.index[0]
print()
for tasa in (0.05, 0.10, 0.20):
    print(f"Recuperar {tasa:.0%} del revenue abandonado en {top_cat}: "
          f"${prize.loc[top_cat, 'revenue_en_juego']*tasa:,.0f}")

# (b) VALOR de mover un one-time al núcleo recurrente (usa los segmentos de 4.6)
gap = rev_repeat / repeat - rev_one_time / one_time
print(f"\nTicket one-time ${rev_one_time/one_time:,.0f} vs recurrente ${rev_repeat/repeat:,.0f} "
      f"(brecha ${gap:,.0f}/usuario)")
print(f"Si el 5% de los {one_time:,} one-time se vuelve recurrente: "
      f"${0.05*one_time*gap:,.0f} de revenue incremental")

# ── Gráfico: revenue en juego por categoría (barh), categoría top resaltada ──
cb = sns.color_palette('colorblind')
pr = prize.sort_values('revenue_en_juego')
colores = [cb[0] if c == top_cat else '#b0b0b0' for c in pr.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(pr.index, pr['revenue_en_juego'], color=colores, edgecolor='white')
ax.set_title('Revenue en juego en carritos abandonados, por categoría', fontweight='bold')
ax.set_xlabel('Revenue abandonado (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}k'))
plt.tight_layout()
plt.show()

**Premio en $ — la fuga vale casi tanto como lo que se vende.**

Sobre toda la data hay **$489,8 M en juego en carritos abandonados** (1.660.501 unidades). No es
una fuga de cola: **electronics concentra $349,5 M (71%)** del revenue abandonado, y con appliances
($39,4 M) y computers ($30,0 M) suma el **~86%**. El ticket del carrito abandonado es alto
(electronics $424, computers $428), coherente con categorías de alta consideración.

Esto pone número a la palanca del persuadible (Hallazgos 1 y 5): **recuperar solo el 10% del
revenue abandonado en electronics = $34,9 M** (al 20%, $69,9 M). Aun con tasas modestas, el retorno
es de ocho cifras.

La segunda palanca, retención (Hallazgo 6), es de magnitud comparable: la brecha de valor
one-time→recurrente es de **$1.149/usuario**; **convertir el 5% de los 440.511 one-time en
recurrentes = $25,3 M** incrementales.

**Traducción de negocio:** las dos grandes bolsas de valor son (1) **recuperar carritos de
electrónica** y (2) **fidelizar al one-time hacia el núcleo recurrente**. El incentivo debe
priorizar electrónica por el tamaño y la concentración del premio.

*(Caveat: "revenue en juego" asume 1 unidad por carrito y no toda recuperación es factible —de ahí
los escenarios 5/10/20%.)*

In [ ]:
# (b-timing) Para recurrentes: días entre la 1a y la 2a OCASIÓN de compra (cuándo disparar
# el nudge) y si la 2a compra es de la MISMA categoría o cross-sell. Sobre purchases_pd.
occ = (purchases_pd.groupby(['user_id', 'user_session'])
       .agg(t=('event_time', 'min'),
            category=('macro_category', lambda s: s.dropna().mode().iloc[0] if s.dropna().size else pd.NA))
       .reset_index()
       .sort_values(['user_id', 't']))
occ['rank'] = occ.groupby('user_id').cumcount() + 1

first  = occ[occ['rank'] == 1].set_index('user_id')[['t', 'category']]
second = occ[occ['rank'] == 2].set_index('user_id')[['t', 'category']]
pair = first.join(second, lsuffix='_1', rsuffix='_2', how='inner')   # usuarios con >=2 ocasiones
days = (pair['t_2'] - pair['t_1']).dt.total_seconds() / 86400

print(f"Usuarios con >=2 ocasiones (recurrentes): {len(pair):,}")
print(f"Días a la 2a compra — mediana {days.median():.1f}, media {days.mean():.1f}")
print(f"  <=1 día: {(days<=1).mean()*100:.1f}%  |  <=7 días: {(days<=7).mean()*100:.1f}%  |  "
      f"<=14 días: {(days<=14).mean()*100:.1f}%")

cat_pair = pair.dropna(subset=['category_1', 'category_2'])
cat_pair = cat_pair[(cat_pair['category_1'] != 'Unknown') & (cat_pair['category_2'] != 'Unknown')]
same = (cat_pair['category_1'] == cat_pair['category_2']).mean() * 100
print(f"\n2a compra MISMA categoría: {same:.1f}%  |  cross-sell (distinta): {100-same:.1f}%  "
      f"(sobre {len(cat_pair):,} pares con categoría conocida)")

# ── Gráfico: distribución de días a la 2a compra (cap 30 días) ──
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(days[days <= 30], bins=30, color=sns.color_palette('colorblind')[0], edgecolor='white')
ax.axvline(days.median(), color='#d62728', ls='--', lw=1.8, label=f'Mediana: {days.median():.1f} días')
ax.set_title('¿Cuándo vuelve el comprador? Días entre 1a y 2a compra', fontweight='bold')
ax.set_xlabel('Días hasta la 2a compra')
ax.set_ylabel('Usuarios recurrentes')
ax.legend()
plt.tight_layout()
plt.show()

**Timing de retención — el que vuelve, vuelve rápido y a lo mismo.**

De los 256.959 recurrentes, la 2ª compra llega **pronto**: mediana de **2.7 días**, con el
**39.1% dentro del primer día** y el **65.3% dentro de la primera semana** (78.8% en 14 días).
No es un retorno mensual: la ventana de reactivación es de días.

Y la 2ª compra es casi siempre **de la misma categoría (82.5%)**; el cross-sell (categoría
distinta) es minoritario (17.5%, sobre 164.695 pares con categoría conocida). El patrón es de
continuidad/reposición dentro de la misma vertical, no de salto a otra.

**Traducción de negocio:** la retención no se juega semanas después sino en los **primeros días
tras la compra**, y el gancho correcto es **más de lo mismo** (misma categoría), no un cross-sell.

**Acción:** disparar el nudge de recompra (email/push, recomendación) en las **primeras 24–72 h**
post-compra, con producto de la **misma categoría** que el usuario acaba de comprar.

In [ ]:
# (c) MARCA dentro de electronics (donde está el premio). marca_elec viene de la capa de
# agregados: unidades que llegan a carrito, agrupadas por marca, >=100 carritos, sin
# "Unknown" (el placeholder de marca imputado en Silver se reporta aparte).
foco = 'electronics'
print(f"{foco}: {n_elec_total:,} carritos; sin marca (Unknown): "
      f"{n_elec_nobrand:,} ({n_elec_nobrand/n_elec_total*100:.1f}%)")
print(f"\nMarcas en {foco} con >=100 carritos (orden: carritos abandonados):")
print(marca_elec.head(10))

# ── Gráfico: top-10 marcas por carritos abandonados en electronics ──
top = marca_elec.head(10).sort_values('abandonados')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top.index, top['abandonados'], color=sns.color_palette('colorblind')[0], edgecolor='white')
ax.set_title(f'{foco}: carritos abandonados por marca (top 10)', fontweight='bold')
ax.set_xlabel('Carritos abandonados')
for y, (ab, rate) in enumerate(zip(top['abandonados'], top['abandono_%'])):
    ax.text(ab + top['abandonados'].max() * 0.01, y, f'{rate:.0f}% aband.',
            va='center', fontsize=8, color='#555')
plt.tight_layout()
plt.show()

**Marca — la intención de electrónica es Samsung y Apple; el dinero abandonado es Apple.**

Dentro de electronics la marca casi siempre está presente (solo **0.6%** sin marca). La intención
se concentra en dos marcas: **Samsung (587.675 carritos) + Apple (550.213) = ~68%** de los carritos
de electronics; con Xiaomi llegan al **~81%**. La tasa de abandono es relativamente pareja
(~47–57%), así que el diferenciador no es el cierre sino el **volumen y el ticket**.

Y ahí Apple destaca: su ticket (**$746**) triplica al de Samsung (**$230**), de modo que aunque
abandona un volumen similar (270.419 vs 273.969 carritos), **el revenue abandonado de Apple
(~$202 M) es ~3× el de Samsung (~$63 M)** y representa cerca de la **mitad de TODO el revenue
abandonado de electronics** ($349,5 M). Samsung aporta volumen; Apple aporta valor.

**Traducción de negocio:** el "persuadible de electrónica" tiene nombre de marca — **Apple y
Samsung** concentran la oportunidad: Apple por valor (ticket alto), Samsung por volumen de carritos.

**Acción:** priorizar la recuperación de carritos **Apple** (mayor $ por carrito) y **Samsung**
(mayor volumen) en electronics; son el blanco más rentable del incentivo de cierre.

*(Caveat: el revenue abandonado por marca es aproximado = nº de abandonados × ticket mediano de la marca.)*

In [ ]:
# (d) PARETO: concentración del revenue (suma de price de eventos purchase). Sobre purchases_pd.
rev_prod = purchases_pd.groupby('product_id')['price'].sum().sort_values(ascending=False)
rev_total_p = rev_prod.sum()
cum = rev_prod.cumsum() / rev_total_p
n_prod = len(rev_prod)

for thr in (0.5, 0.8, 0.9):
    k = int((cum <= thr).sum()) + 1
    print(f"El {thr:.0%} del revenue lo hacen {k:,} productos "
          f"({k/n_prod*100:.1f}% de los {n_prod:,} productos con venta)")

rev_cat_pct = (purchases_pd[purchases_pd['macro_category'] != 'Unknown']
               .groupby('macro_category')['price'].sum()
               .sort_values(ascending=False) / rev_total_p * 100).round(1)
print("\nRevenue por categoría (% del total; 'Unknown' excluido):")
print(rev_cat_pct.head(8))
print(f"Top 3 categorías = {rev_cat_pct.head(3).sum():.1f}% del revenue")

# ── Gráfico: curva de Pareto de productos ──
x = (pd.RangeIndex(1, n_prod + 1) / n_prod) * 100
k80 = int((cum <= 0.8).sum()) + 1
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, cum.values * 100, color=sns.color_palette('colorblind')[0], lw=2)
ax.axhline(80, color='#d62728', ls='--', lw=1, label='80% del revenue')
ax.axvline(k80 / n_prod * 100, color='#d62728', ls=':', lw=1,
           label=f'{k80/n_prod*100:.0f}% de los productos')
ax.set_title('Pareto: concentración del revenue por producto', fontweight='bold')
ax.set_xlabel('% de productos (de mayor a menor revenue)')
ax.set_ylabel('% acumulado del revenue')
ax.legend()
plt.tight_layout()
plt.show()

**Pareto — el negocio es un puñado de productos y, sobre todo, electrónica.**

La concentración es extrema en los dos ejes. Por producto: **el 1% de los productos genera el 80%
del revenue** (677 de 68.076), y apenas **55 productos (0.1%) hacen la mitad**. Por categoría:
**electronics concentra el 75.6% del revenue**, y el top-3 (electronics, appliances, computers)
suma el **87%**. El resto es cola larga marginal en ingresos.

**Traducción de negocio:** no conviene repartir el esfuerzo por todo el catálogo. Una cabeza
pequeña de productos —y casi un solo vertical, electrónica— mueve la aguja. Esto cierra el círculo
con todo lo anterior: electrónica es donde se concentran la intención (Hallazgo 1), el abandono y
su revenue (Hallazgo 5 + premio), las marcas clave (Apple/Samsung) y también el revenue total.

**Acción:** enfocar incentivos, merchandising y recuperación de carrito en el *head* de productos
de **electronics** (con Apple/Samsung al frente); es donde cada punto de mejora rinde más.

### **4.9 Síntesis de Hallazgos**

Leídos en conjunto, los siete hallazgos y los cuatro análisis de oportunidad dibujan un mapa de
decisión con **dos palancas de valor de magnitud comparable**, ambas concentradas en un mismo terreno.

**El terreno: el negocio es electrónica.**
La conversión global por unidad es **2.23%**, pero el revenue está brutalmente concentrado: el
**1% de los productos genera el 80%** y **electronics concentra el 75.6% del revenue** (top-3
categorías = 87%). El precio no explica la conversión —las categorías más baratas (apparel,
accessories) están entre las que peor cierran—, así que el descuento no es la palanca por defecto.
El cierre es parejo (~43–51%) entre categorías; lo que distingue a electrónica es el **volumen**.

---

**Palanca A — Conversión del persuadible (Hallazgos 1 y 5; marca; premio en $).**
Existe un segmento grande de intención declarada que no cierra: el **51.7% de los carritos se
abandona**, y la mitad de ese pool está en **electronics** (824.508 carritos, **$349,5 M en juego**
— el 71% de todo el revenue abandonado). Tiene nombre de marca: **Samsung + Apple ≈ 68% de los
carritos** de electrónica, y Apple, por su ticket alto ($746), concentra **~la mitad del revenue
abandonado** de la categoría. Recuperar apenas el 10% de ese abandono en electronics vale **$34,9 M**.

*Cuándo y cómo:* las compras pican a **media mañana (7–11h, pico 9h)**, franja que además
**convierte ~2× por visita** que la tarde; y la decisión es casi inmediata (**mediana 2.2 min**,
91% en <10 min). El incentivo debe ser **inmediato y en pantalla** (recordatorio de carrito,
urgencia/stock, financiación), no diferido ni de precio.

**Palanca B — Retención del núcleo recurrente (Hallazgo 6; timing).**
El **36.8% de los compradores vuelve** a comprar y ese núcleo concentra el **73.8% del revenue**,
con un ticket casi **5× el del one-time** ($1.450 vs $301); la brecha de valor de mover un one-time
a recurrente es de **$1.149/usuario**.

*Cuándo y cómo:* el que vuelve, vuelve **rápido y a lo mismo** — mediana **2.7 días**, 65% en la
primera semana, y el **82.5% repite en la misma categoría**. La acción es un **nudge de recompra
en las primeras 24–72 h**, recomendando más de la misma categoría.

---

**Mensaje central (dos palancas, dos públicos, dos acciones):**

> **El negocio se juega en electrónica, en dos frentes de igual peso: (A) recuperar los carritos
> de alta intención que no cierran —Samsung/Apple— con incentivo inmediato en la mañana, y (B)
> retener al núcleo recurrente que ya aporta ~3 de cada 4 dólares, con un nudge de recompra a los
> pocos días en su misma categoría.**

La Fase II construye el argumento visual aclaratorio de estas dos palancas: a quién, dónde, cuándo
y con qué incentivar.

### **4.10 Comparativa: cómo cambió cada hallazgo al pasar a la data completa**

Este EDA pasó por **dos saltos de fuente**: (1) del muestreo *por evento* al muestreo *por usuario*
del taller, y (2) —en este notebook— de esa muestra (≈2,3% de octubre) a las capas **Silver/Gold
completas** del PI (**Oct+Nov, 109.5M eventos / 23M sesiones**). La tabla resume **muestra (taller)
→ data completa (PI)**:

| # | Hallazgo | Muestra por usuario (taller) | Data completa (PI) | ¿Se sostiene? |
|---|----------|------------------------------|--------------------|----------------|
| 1 | Funnel por unidad | conv 2.44%; abandono ~32% | conv **2.23%**; abandono **51.7%** | Estructura sí; el cierre real es más bajo |
| 2 | Precio no es el freno | los baratos convierten peor | **se confirma** (electronics $146 lidera; apparel/accessories peores) | ✔ robusto |
| 3 | Ventana horaria | pico 7h; mitad de semana | pico **9h**; **fin de semana** lidera en volumen | Mañana sí; el día se invirtió |
| 4 | Horaria por categoría | apparel 12h; furniture 6h | casi todo a **9h**; **apparel 17h** | Afinado |
| 5 | Abandono de carrito | ~32%, 67% en electronics | **51.7%**; electronics = **50% del pool** (mejor cierre, más volumen) | Magnitud ↑; foco electrónica ✔ |
| 6 | One-time vs recurrentes | 31.7% recurrentes; 69% revenue | **36.8%** recurrentes; **73.8%** revenue; ticket ~5× | ✔ y reforzado |
| 7 | Velocidad de decisión | mediana 2.2 min | mediana **2.2 min**; 91% <10 min | ✔ idéntico |

**Lectura:** los hallazgos **2, 6 y 7 se sostienen** (incluso se refuerzan); el **1 y el 5** mantienen
la dirección (electrónica es el terreno) pero el **abandono real es ~52%, no ~32%**; el **3 y el 4**
se afinan (pico a 9h, fin de semana lidera, apparel vespertino). La conclusión de negocio —dos
palancas concentradas en electrónica— **es estable a escala completa**.

**Análisis de oportunidad (sección 4.8) sobre data completa:**

| Análisis | Resultado clave | Para qué decisión |
|----------|-----------------|-------------------|
| Premio en $ | **$489,8 M** en juego (71% en electronics); recuperar 10% en electronics = **$34,9 M** | Dimensiona la **Palanca A** |
| Timing de retención | 2ª compra: mediana **2.7 días**, 65% en la 1ª semana, **82.5%** misma categoría | **Cuándo y qué** del nudge (Palanca B) |
| Marca (electronics) | **Samsung+Apple ≈ 68%** de los carritos; Apple ($746 ticket) ≈ ½ del revenue abandonado | **A quién** apuntar |
| Pareto | **1% de los productos = 80%** del revenue; electronics = **75.6%** | Enfocar el esfuerzo en el *head* |

## **5. Perfilado de la Gold v1 (grano = sesión · insumo de modelado y tablero)**

Las secciones 1–4 son **nivel evento / producto-en-sesión** (Silver). Esta sección perfila
la **Gold v1** —la tabla que alimenta a Sara (features/modelo) y a Kelly (tablero)— cuyo
grano es **1 fila = 1 `user_session`** y cuya etiqueta es `target_purchase` (¿la sesión
contiene `purchase`?). Es el insumo directo para decidir features y para el contrato Gold (§13).

**Features (anti-fuga, solo comportamiento previo al primer cart/purchase):**
`total_views`, `distinct_products_viewed`, `brands_compared`, `categories_explored`,
`browsing_duration_sec`, y el flag `sin_navegacion_previa`.

> La estadística de etiqueta y del flag se calcula **exacta sobre la Gold completa**; las
> **distribuciones** se grafican sobre una **muestra del 3%** (semilla fija) para no traer
> 23M filas al driver (disciplina de cuota).

In [ ]:
# ── Perfilado exacto de la Gold v1 (sobre la tabla completa) ──────────────────
FEATS = ["total_views", "distinct_products_viewed", "brands_compared",
         "categories_explored", "browsing_duration_sec"]

_g = gold.agg(F.count("*").alias("n"),
              F.sum("target_purchase").alias("pos"),
              F.avg("target_purchase").alias("tasa")).first()
print(f"Gold: {_g['n']:,} sesiones | positivas (compra): {_g['pos']:,} | "
      f"tasa etiqueta: {_g['tasa']:.4f}")

# Flag sin_navegacion_previa (§13): tamaño y tasa de etiqueta. Estas sesiones "abren" con
# cart/purchase → todas las features de navegación en 0; decisión de tratamiento la toma Sara.
print("\nFlag sin_navegacion_previa (corte anti-fuga):")
flag_tab = (gold.groupBy("sin_navegacion_previa").agg(
    F.count("*").alias("sesiones"),
    F.avg("target_purchase").alias("tasa_etiqueta")).orderBy("sin_navegacion_previa").toPandas())
flag_tab["%_sesiones"] = (flag_tab["sesiones"] / _g["n"] * 100).round(3)
print(flag_tab.to_string(index=False))

# Estadísticos de features por clase (target 0/1), exacto.
gold_feat_by_target = (gold.groupBy("target_purchase").agg(
    F.count("*").alias("n"),
    *[F.avg(c).alias(f"avg_{c}") for c in FEATS],
    *[F.expr(f"percentile_approx({c}, 0.5)").alias(f"med_{c}") for c in FEATS],
).toPandas().set_index("target_purchase"))
print("\nFeatures por clase (0 = no compra, 1 = compra):")
print(gold_feat_by_target.T.round(2))

# Muestra del 3% para distribuciones (shapes), semilla fija.
gold_pd = gold.sample(False, 0.03, seed=42).toPandas()
print(f"\ngold_pd (muestra 3%): {gold_pd.shape[0]:,} sesiones × {gold_pd.shape[1]} columnas")

In [ ]:
# ── Distribución de las features por clase (muestra 3%) ───────────────────────
# Solo sesiones CON navegación previa (las sin_navegacion_previa tienen todo en 0 y se
# analizan aparte vía el flag). Densidad normalizada para comparar formas pese al desbalanceo.
nav = gold_pd[~gold_pd["sin_navegacion_previa"]]
cb = sns.color_palette('colorblind')

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Gold v1 — distribución de features por clase (muestra 3%, solo sesiones con navegación)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, c in enumerate(FEATS):
    cap = nav[c].quantile(0.99)
    a = nav.loc[nav["target_purchase"] == 0, c].clip(upper=cap)
    b = nav.loc[nav["target_purchase"] == 1, c].clip(upper=cap)
    axes[i].hist(a, bins=40, density=True, color='#b0b0b0', alpha=0.7, label='No compra (0)')
    axes[i].hist(b, bins=40, density=True, color=cb[0], alpha=0.6, label='Compra (1)')
    axes[i].set_title(c, fontweight='bold')
    axes[i].set_xlabel(f'{c} (cap p99)')
    axes[i].set_ylabel('densidad')
    axes[i].legend(fontsize=8)

# Panel 6: tasa de etiqueta según el flag sin_navegacion_previa (insumo §13).
ax = axes[5]
bars = ax.bar(flag_tab["sin_navegacion_previa"].astype(str),
              flag_tab["tasa_etiqueta"] * 100,
              color=[cb[2], cb[3]], edgecolor='white')
ax.set_title('Tasa de compra según sin_navegacion_previa', fontweight='bold')
ax.set_xlabel('sin_navegacion_previa')
ax.set_ylabel('% de sesiones que compran')
for bar, t, n in zip(bars, flag_tab["tasa_etiqueta"], flag_tab["sesiones"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{t*100:.1f}%\n({n:,} ses.)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlación entre features (muestra 3%) — insumo de multicolinealidad (paso 5) ──
corr = gold_pd[FEATS].corr()
print("Matriz de correlación (Pearson) entre features de la Gold:")
print(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlación entre features (Gold v1, muestra 3%)', fontweight='bold')
plt.tight_layout()
plt.show()

# Correlación de cada feature con la etiqueta (point-biserial ≈ Pearson con 0/1).
corr_y = gold_pd[FEATS + ["target_purchase"]].corr()["target_purchase"].drop("target_purchase")
print("\nCorrelación de cada feature con target_purchase:")
print(corr_y.round(3).sort_values(ascending=False).to_string())

**Hallazgos de la Gold v1 (insumo para Sara y Kelly)**

- **Desbalanceo:** la etiqueta `target_purchase` ≈ **6.1%** (1 de cada ~16 sesiones compra).
  Justifica PR-AUC + calibración como métrica (no accuracy) — coherente con doc 00 §5.2.
- **Señal de las features:** las sesiones que compran muestran más `total_views`,
  `distinct_products_viewed`, `categories_explored` y mayor `browsing_duration_sec` (ver
  medianas por clase) → la **intensidad de navegación** separa compra de no-compra.
- **Multicolinealidad (paso 5):** `total_views`, `distinct_products_viewed` y
  `browsing_duration_sec` tienden a co-moverse (ver heatmap) → revisar antes de modelar.

> **⚠️ Decisión pendiente para Sara — flag `sin_navegacion_previa` (doc 00 §13).** Estas
> sesiones "abren" con `cart`/`purchase`: por el corte anti-fuga tienen **todas las features
> de navegación en 0** y son **mayormente positivas** (ver el panel de tasa por flag). Sara
> decide: (A) mantenerlas usando el flag como variable; (B) excluirlas del entrenamiento;
> o (C) tratarlas como segmento aparte. Impacto chico (~0.15%) pero a definir antes de entrenar.

## **6. Análisis adicionales habilitados por la data completa (Oct+Nov)**

La muestra del taller cubría solo octubre; con **Silver/Gold completas (Oct+Nov)** se abren tres
análisis que complementan la síntesis (§4.9) y sirven directamente a la Pregunta de Oro:

- **6.1 Evolución temporal (Oct vs Nov):** ¿el funnel es estable o cambia con la temporada
  (Black Friday)? Es la **evidencia para revisar el split train/test** (decisión abierta con Sara).
- **6.2 Curva de intención:** ¿cómo crece la probabilidad de compra con la intensidad de navegación?
  Puente entre el funnel y las features del modelo (insumo de Sara y del tablero).
- **6.3 Tipología de visitantes:** segmenta a **todos** los visitantes (no solo compradores) en
  browser / intender / buyer — responde el "**qué segmento**" de la Pregunta de Oro (versión ligera
  por reglas; el clustering formal es de Sara).

> **Nota metodológica (anti-fuga).** Todo aquí es **descriptivo** sobre Oct+Nov. Noviembre es el
> periodo de **prueba** del modelo (split temporal, hoy en revisión — ver §6.1): **no** se derivan
> features ni decisiones de modelado a partir de noviembre.
>
> **Nota de versión.** Este EDA es **v1 sobre Silver/Gold actuales**. Si la pasada 2 (§2.3.1 paso 5)
> limpia bots/outliers/`"Unknown"`, varios números cambiarán → re-correr (con `FORCE_REBUILD_UNITS=True`
> si cambia Silver) y refrescar la narrativa.

### **6.1 Evolución temporal — Oct vs Nov (evidencia para el split train/test)**

Serie diaria de volumen e intensidad de compra, y el funnel por unidad comparado entre meses.
Si Oct y Nov difieren mucho (Black Friday), el split **train=Oct / test=Nov** estaría midiendo el
modelo bajo un cambio de régimen → es la evidencia para decidir, con Sara, el corte temporal definitivo.

In [ ]:
# 6.1 Evolución temporal (Oct vs Nov). Habilitado por la data completa.
# (a) Serie diaria: eventos por tipo + intensidad de compra (1 pasada a Silver).
daily = (silver.groupBy("date").pivot("event_type", ["view", "cart", "purchase"]).count()
         .orderBy("date").toPandas())
daily = daily.rename(columns={"view": "views", "cart": "carts", "purchase": "purchases"}).fillna(0)
daily["date"] = pd.to_datetime(daily["date"])
daily["conv_x100"] = daily["purchases"] / daily["views"] * 100

# (b) Funnel por UNIDAD por mes (reconstrucción de unidades dentro de cada mes).
units_m = (silver.withColumn("mes", F.date_format("event_time", "yyyy-MM"))
           .groupBy("mes", "user_session", "product_id").agg(
               F.max(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("hc"),
               F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("hp")))
month_funnel = (units_m.groupBy("mes").agg(
    F.count("*").alias("units"),
    F.sum(F.when((F.col("hc") == 1) | (F.col("hp") == 1), 1).otherwise(0)).alias("reached_cart"),
    F.sum("hp").alias("purchased")).orderBy("mes").toPandas().set_index("mes"))
month_funnel["cart_rate"] = month_funnel["reached_cart"] / month_funnel["units"] * 100
month_funnel["conv_rate"] = month_funnel["purchased"] / month_funnel["units"] * 100
month_funnel["abandono"]  = (1 - month_funnel["purchased"] / month_funnel["reached_cart"]) * 100
print("Volumen de eventos por mes:")
print(daily.assign(mes=daily["date"].dt.strftime("%Y-%m")).groupby("mes")[["views", "carts", "purchases"]].sum())
print("\nFunnel por unidad, por mes:")
print(month_funnel.round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# Izq: serie diaria (compras + intensidad)
ax0, ax0b = axes[0], axes[0].twinx()
ax0.bar(daily["date"], daily["purchases"], color="#b0d0e8", width=0.9)
ax0b.plot(daily["date"], daily["conv_x100"], color="#d62728", lw=1.8)
ax0.set_title("Serie diaria: compras (barras) e intensidad compras/100 vistas (línea)", fontweight="bold")
ax0.set_ylabel("Compras/día"); ax0b.set_ylabel("Compras por 100 vistas", color="#d62728")
ax0.tick_params(axis="x", rotation=45)
# Der: funnel por mes
m = month_funnel.reset_index()
x = range(len(m)); w = 0.25
axes[1].bar([i - w for i in x], m["cart_rate"], width=w, label="Cart rate %")
axes[1].bar([i for i in x],       m["conv_rate"], width=w, label="Conv rate %")
axes[1].bar([i + w for i in x], m["abandono"],  width=w, label="Abandono %")
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(m["mes"])
axes[1].set_title("Funnel por unidad: Oct vs Nov", fontweight="bold")
axes[1].set_ylabel("%"); axes[1].legend()
plt.tight_layout()
plt.show()

### **6.2 Curva de intención — conversión según intensidad de navegación**

¿Cómo crece la probabilidad de compra de la sesión con su intensidad de navegación? Tasa de compra
por decil de `total_views`, `distinct_products_viewed` y `browsing_duration_sec` (Gold, muestra 3%,
sesiones con navegación). Si la curva es monótona creciente, esas features tienen **poder predictivo**
(insumo de Sara) y dan el mensaje "más exploración → más compra" para el tablero (Kelly).

In [ ]:
# 6.2 Curva de intención (Gold, muestra 3%). Solo sesiones con navegación previa.
nav = gold_pd[~gold_pd["sin_navegacion_previa"]]
feats_curve = ["total_views", "distinct_products_viewed", "browsing_duration_sec"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Curva de intención: % de sesiones que compran según intensidad de navegación',
             fontsize=13, fontweight='bold')
for ax, f in zip(axes, feats_curve):
    q = pd.qcut(nav[f], 10, duplicates="drop")
    conv = nav.groupby(q, observed=True)["target_purchase"].mean() * 100
    mids = [iv.mid for iv in conv.index]
    ax.plot(mids, conv.values, marker="o", color=sns.color_palette("colorblind")[0])
    ax.axhline(nav["target_purchase"].mean() * 100, color="gray", ls="--", lw=1,
               label=f"Media: {nav['target_purchase'].mean()*100:.1f}%")
    ax.set_title(f"Conversión vs {f}", fontweight="bold")
    ax.set_xlabel(f"{f} (valor central del decil)")
    ax.set_ylabel("% que compra")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Conversión por decil de total_views (sesiones con navegación):")
_q = pd.qcut(nav["total_views"], 10, duplicates="drop")
print((nav.groupby(_q, observed=True)["target_purchase"].mean() * 100).round(2))

### **6.3 Tipología de visitantes — browser / intender / buyer (nivel sesión)**

Segmenta **todas** las sesiones (no solo las que compran) según hasta dónde llegan en el funnel:
**browser** (solo vistas), **intender** (llega a carrito, no compra) y **buyer** (compra). Es el
funnel al **grano de la etiqueta** (sesión) y responde el "qué segmento" de la Pregunta de Oro.
El `buyer` debe casar con la tasa de etiqueta de la Gold (~6.1%). Versión **ligera por reglas**;
el clustering formal (k-means, perfilado) es del frente de Sara.

In [ ]:
# 6.3 Tipología de visitantes a nivel SESIÓN (desde UNIT + perfil de engagement desde Gold).
sess = units.groupBy("user_session").agg(
    F.max(F.when(F.col("has_cart") | F.col("has_purchase"), 1).otherwise(0)).alias("rc"),
    F.max(F.col("has_purchase").cast("int")).alias("p"))
sess = sess.withColumn("segmento",
                       F.when(F.col("p") == 1, "buyer")
                        .when(F.col("rc") == 1, "intender")
                        .otherwise("browser"))

# Perfil de engagement por segmento (join con Gold por user_session).
seg = (sess.join(gold.select("user_session", "total_views", "distinct_products_viewed",
                             "browsing_duration_sec"), "user_session", "left")
       .groupBy("segmento").agg(
           F.count("*").alias("sesiones"),
           F.avg("total_views").alias("avg_views"),
           F.avg("distinct_products_viewed").alias("avg_prod"),
           F.avg("browsing_duration_sec").alias("avg_dur_seg"))
       .toPandas().set_index("segmento").reindex(["browser", "intender", "buyer"]))
seg["%_sesiones"] = (seg["sesiones"] / seg["sesiones"].sum() * 100).round(2)
print("Tipología de visitantes (nivel sesión):")
print(seg.round(2))
print(f"\nRevenue total capturado (compras): ${purchases_pd['price'].sum():,.0f} "
      f"— todo en el segmento 'buyer' ({seg.loc['buyer','%_sesiones']}% de las sesiones).")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cols = sns.color_palette("colorblind", 3)
axes[0].bar(seg.index, seg["sesiones"], color=cols)
axes[0].set_title("Sesiones por segmento", fontweight="bold")
axes[0].set_ylabel("Sesiones")
axes[0].set_ylim(0, seg["sesiones"].max() * 1.15)
for i, (n, p) in enumerate(zip(seg["sesiones"], seg["%_sesiones"])):
    axes[0].text(i, n, f"{n:,.0f}\n({p}%)", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].bar(seg.index, seg["avg_views"], color=cols)
axes[1].set_title("Engagement medio (total_views) por segmento", fontweight="bold")
axes[1].set_ylabel("total_views medio")
for i, v in enumerate(seg["avg_views"]):
    axes[1].text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

## **FASE II - Composición del mensaje**

### 🎯 Gráfico héroe — Palanca A: el premio abandonado en electrónica

**Pregunta de negocio (PN1):** ¿dónde perdemos conversión y cuánto vale recuperarla?

De los siete hallazgos, el que cuenta la tesis de un solo golpe es el **revenue en juego en carritos abandonados** (4.8a): traduce el problema a **dinero** y a una **acción** clara, y concentra el premio en una sola categoría. Es el candidato a héroe del dashboard (decisión D4).

A continuación, la **transición exploratorio → aclaratorio** del mismo gráfico (lo que evalúa la Fase 2 de la rúbrica): primero la versión de análisis, luego la versión diseñada para que un gerente decida en 30 segundos, y la justificación de cada decisión visual.

In [ ]:
# ── FASE II · ANTES: versión EXPLORATORIA del héroe (reproduce 4.8a) ──
# Reusa `prize` y `top_cat` de 4.8a (revenue en juego por categoría, por unidad).
pr = prize.sort_values('revenue_en_juego')
col0 = ['#1f4e8c' if c == top_cat else '#b0b0b0' for c in pr.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(pr.index, pr['revenue_en_juego'], color=col0, edgecolor='white')
ax.set_title('Revenue en juego en carritos abandonados, por categoría', fontweight='bold')
ax.set_xlabel('Revenue abandonado (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}k'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── FASE II · DESPUÉS: versión ACLARATORIA del héroe (Palanca A) ──
# Reusa `prize` y `top_cat` de 4.8a (sobre Silver/Gold completas).
def _money(v):
    return f"${v/1e6:.1f} M".replace('.', ',') if v >= 1e6 else f"${v/1e3:.0f}k"
def _money_full(v):
    return f"${v:,.0f}".replace(',', '.')

total_prize = prize['revenue_en_juego'].sum()
elec_prize  = prize.loc[top_cat, 'revenue_en_juego']
pct_elec    = elec_prize / total_prize
rec10, rec20 = elec_prize * 0.10, elec_prize * 0.20

# Data-to-Ink: top-3 categorías + resto colapsado (no compite por la mirada)
top3  = prize.head(3)['revenue_en_juego']
resto = prize.iloc[3:]['revenue_en_juego']
acl = top3.copy()
if len(resto):
    acl[f"resto ({len(resto)} cat.)"] = resto.sum()
acl = acl.sort_values()                      # mayor arriba en barh

ACCENT, GREY = '#1f4e8c', '#cfcfcf'
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.barh(acl.index, acl.values, color=[ACCENT if c == top_cat else GREY for c in acl.index])

# Etiquetas $ directas (permiten eliminar el eje X) + callout % en la barra clave
for y, (cat, val) in enumerate(acl.items()):
    ax.text(val + total_prize * 0.012, y, _money(val), va='center', fontsize=11,
            fontweight='bold' if cat == top_cat else 'normal',
            color='#1a1a1a' if cat == top_cat else '#8a8a8a')
yi = list(acl.index).index(top_cat)
ax.text(elec_prize * 0.5, yi, f"{pct_elec:.0%} del premio", va='center', ha='center',
        color='white', fontweight='bold', fontsize=12)

# Data-to-Ink: fuera eje X, spines y grid; resaltar solo la etiqueta clave
ax.set_xlim(0, elec_prize * 1.2)
ax.xaxis.set_visible(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(axis='y', length=0)
for lbl in ax.get_yticklabels():
    if lbl.get_text() == top_cat:
        lbl.set_fontweight('bold'); lbl.set_color('#1a1a1a')
    else:
        lbl.set_color('#8a8a8a')

# Jerarquía + acto de habla directivo: título-acción y subtítulo de contexto
ax.set_title(f"Recuperar el 10% de los carritos de electrónica = {_money_full(rec10)}",
             fontsize=15, fontweight='bold', loc='left', pad=30)
ax.annotate(f"El {pct_elec:.0%} del revenue abandonado está en una sola categoría: {top_cat}",
            xy=(0, 1.04), xycoords='axes fraction', fontsize=11, color='#555')

# Recorrido visual: anotación que lleva la mirada del premio a la acción
ax.annotate(f"Recuperar 20% → {_money_full(rec20)}",
            xy=(elec_prize, yi), xytext=(elec_prize * 0.60, yi - 1.5),
            fontsize=10.5, color=ACCENT, fontweight='bold', ha='left', va='center',
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.6),
            bbox=dict(boxstyle='round,pad=0.4', fc='#eef3fb', ec=ACCENT, lw=1.2))

fig.text(0.01, -0.04,
    "Fuente: Silver/Gold v1 completas (Oct+Nov 2019; 109,5 M eventos). “Revenue en juego” = precio de "
    "unidades (producto-en-sesión) que llegaron a carrito y no compraron; asume 1 unidad/carrito y "
    f"recuperación parcial (escenarios 5/10/20%). Premio total: {_money(total_prize)}.",
    fontsize=8, color='#9a9a9a')
plt.tight_layout()
plt.show()

### De exploratorio a aclaratorio — justificación de las decisiones visuales

La versión exploratoria (arriba) es correcta para el analista, pero obliga a *leer* el gráfico: recorrer el eje, comparar las 10 categorías y deducir la conclusión. La versión aclaratoria está diseñada para **convencer y motivar una acción** (acto de habla directivo) de un vistazo.

| Decisión de diseño | Principio del curso | Antes (exploratoria) → Después (aclaratoria) |
|---|---|---|
| **Título-acción** con verbo y cifra ("Recuperar el 10% … = \$207.700") | Acto de habla directivo · llamada a la acción (5ª capa del argumento) | Título descriptivo ("Revenue en juego…") → título que dice **qué hacer y cuánto vale** |
| **Color solo en electronics**, resto en gris | Atributos preatentivos · contraste (gris = contexto, color = dato clave) | Ya presente; se mantiene y refuerza |
| **Eliminar eje X y grilla**; etiquetas de \$ directas sobre la barra | Data-to-Ink · "menos es más" | Eje en \$k + grilla → **cero tinta no informativa**; el dato se lee en la barra |
| **Colapsar la cola** en "resto (N cat.)" | Data-to-Ink · una gráfica, un mensaje | 10 categorías compitiendo → 3 + resto; la mirada no se dispersa |
| **Callout "82% del premio"** dentro de la barra | Jerarquía · anotación que revela el mensaje | El lector deducía la concentración → se le **dice** |
| **Anotación con flecha** al escenario de recuperación (20% → \$415.400) | Recorrido visual (rompe el patrón Z) · storytelling integrado | Sin acción visible → la mirada va del premio a **la palanca** |
| **Subtítulo de contexto** ("el 82% está en una sola categoría") | Anatomía del argumento: contexto + patrón | — (la conclusión quedaba implícita) |
| **Barras ordenadas por valor** | Balance · reduce fricción de lectura | Se mantiene |

**Patrón Contexto → Hallazgo → Traducción de negocio → Acción:**
- **Contexto:** el negocio pierde ventas en el carrito (abandono real 32,4%).
- **Hallazgo:** el 82% del revenue abandonado (\$2,08 M de \$2,53 M) está en **electronics**.
- **Traducción:** recuperar incluso una fracción modesta vale seis cifras (10% = \$207.700; 20% = \$415.400) **sobre una muestra del 2,3% de octubre** → a escala completa, ~40× más.
- **Acción:** priorizar la recuperación de carritos de electrónica con incentivo inmediato/en pantalla en la franja matutina (ver H3 y H7).

**Prueba de los 30 segundos:** el título dice la acción y el dinero; el color lleva la mirada a electronics; la anotación cuantifica la palanca. El gerente decide *"recuperar carritos de electrónica"* sin leer el eje.

### 🎯 Segundo héroe — Palanca B: el valor está en quien vuelve (PN2)

**Pregunta de negocio (PN2):** ¿quiénes son los clientes valiosos y cómo retenerlos?

El segundo momento de la historia (D2) es la **retención**. El Hallazgo 6, sobre datos completos,
muestra que la recurrencia no es marginal: **el 36.8% de los compradores vuelve y concentra el
73.8% del revenue**. Misma transición exploratorio → aclaratorio.

In [ ]:
# ── FASE II · ANTES (Palanca B): exploratoria — reproduce 4.6 ──
# Reusa one_time / repeat / n_buyers / rev_one_time / rev_repeat / rev_total de 4.6.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
colores = sns.color_palette('colorblind', 2)
labels_seg = ['One-time', 'Recurrentes']
axes[0].bar(labels_seg, [one_time, repeat], color=colores, edgecolor='white', width=0.5)
axes[0].set_title('Número de compradores por segmento', fontweight='bold')
axes[0].set_ylabel('Usuarios')
for i, val in enumerate([one_time, repeat]):
    axes[0].text(i, val + 50, f'{val:,}\n({val/n_buyers*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
axes[1].bar(labels_seg, [rev_one_time, rev_repeat], color=colores, edgecolor='white', width=0.5)
axes[1].set_title('Revenue por segmento (USD)', fontweight='bold')
axes[1].set_ylabel('Revenue')
for i, val in enumerate([rev_one_time, rev_repeat]):
    axes[1].text(i, val + 200, f'${val:,.0f}\n({val/rev_total*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── FASE II · DESPUÉS (Palanca B): aclaratoria — concentración de valor + acción de retención ──
# Reusa one_time/repeat/n_buyers/rev_* de 4.6 y days/same de 4.8b.
def _pct(x):
    return f"{x:.1f}%".replace('.', ',')

pct_b = repeat / n_buyers * 100             # % de compradores recurrentes
pct_r = rev_repeat / rev_total * 100        # % del revenue que generan
ticket_rep = rev_repeat / repeat
ticket_one = rev_one_time / one_time
ratio = ticket_rep / ticket_one
med_days = float(days.median())             # de 4.8b

ACCENT, GREY = '#1f4e8c', '#d9d9d9'
fig, ax = plt.subplots(figsize=(8.5, 5.6))
x = [0, 1]
ax.bar(x, [pct_b, pct_r], width=0.55, color=ACCENT)                                 # recurrentes (base 0)
ax.bar(x, [100 - pct_b, 100 - pct_r], bottom=[pct_b, pct_r], width=0.55, color=GREY) # one-time (arriba)

# Etiquetas % directas: recurrentes en blanco/bold, one-time atenuado
for xi, v in zip(x, [pct_b, pct_r]):
    ax.text(xi, v / 2, _pct(v), ha='center', va='center', color='white', fontweight='bold', fontsize=15)
for xi, v, b in zip(x, [100 - pct_b, 100 - pct_r], [pct_b, pct_r]):
    ax.text(xi, b + v / 2, _pct(v), ha='center', va='center', color='#8a8a8a', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(['Compradores', 'Revenue'], fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.yaxis.set_visible(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)

# Flecha que conecta el mismo grupo recurrente
ax.annotate('', xy=(1, pct_r / 2), xytext=(0, pct_b / 2),
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.8, connectionstyle='arc3,rad=-0.25'))
ax.text(0.5, max(pct_b, pct_r) / 2 + 4, 'el mismo grupo', ha='center', color=ACCENT,
        fontstyle='italic', fontsize=10)

# Título-acción + subtítulo de contexto
ax.set_title('Fideliza al tercio que ya vuelve: genera ~3 de cada 4 dólares',
             fontsize=15, fontweight='bold', loc='left', pad=30)
ratio_s = f"{ratio:.1f}".replace('.', ',')
tr = f"{ticket_rep:,.0f}".replace(',', '.')
to = f"{ticket_one:,.0f}".replace(',', '.')
ax.annotate(f"El {_pct(pct_b)} de los compradores (recurrentes) concentra el {_pct(pct_r)} del revenue · "
            f"ticket {ratio_s}× mayor ({tr} vs {to} USD)",
            xy=(0, 1.04), xycoords='axes fraction', fontsize=10, color='#555')

# Caja de acción (timing del nudge)
md_s = f"{med_days:.1f}".replace('.', ',')
sm_s = f"{same:.1f}".replace('.', ',')
ax.text(1.03, 0.0,
        f"Acción\nVuelven en {md_s} días (mediana)\ny {sm_s}% a la misma categoría\n→ nudge de recompra 24–72 h",
        transform=ax.transAxes, va='bottom', ha='left', fontsize=10, color='#1a1a1a',
        bbox=dict(boxstyle='round,pad=0.5', fc='#eef3fb', ec=ACCENT, lw=1.2))

fig.text(0.01, -0.02,
    "Fuente: Gold v1 completa (Oct+Nov 2019). Recurrente = compra en ≥2 sesiones distintas; "
    "“días a la 2a compra” y “misma categoría” sobre usuarios con ≥2 ocasiones.",
    fontsize=8, color='#9a9a9a')
plt.tight_layout()
plt.show()

### De exploratorio a aclaratorio — Palanca B (retención)

La exploratoria (arriba) usa **dos gráficos** (conteo y revenue) que el lector debe comparar mentalmente para captar la desproporción. La aclaratoria la muestra **en una sola imagen, con base común**, y traduce el hallazgo a una acción con timing.

| Decisión de diseño | Principio del curso | Antes → Después |
|---|---|---|
| **Dos barras 100% apiladas** (Compradores vs Revenue) en lugar de dos gráficos | Gestalt (base común) · una imagen, un mensaje | Comparar 2 gráficos → desproporción de un vistazo |
| **Recurrentes resaltados, anclados en 0** | Atributos preatentivos · contraste; eje común | 31,7% vs 69% comparables a simple vista |
| **Flecha "el mismo grupo"** que conecta 31,7% → 69% | Recorrido visual · storytelling integrado | La relación queda explícita |
| **Título-acción** ("Fideliza al tercio que ya vuelve…") | Acto de habla directivo · llamada a la acción | Títulos descriptivos → qué hacer |
| **Eliminar eje Y y spines**; etiquetas % directas | Data-to-Ink · "menos es más" | Ejes en miles de USD → cero tinta no informativa |
| **Caja de acción con timing** (1,8 días · 85,5% misma categoría → nudge 24–72 h) | Anotación · traducción a acción | El "cuándo/cómo" del incentivo, dentro del gráfico |

**Patrón Contexto → Hallazgo → Traducción de negocio → Acción:**
- **Contexto:** la mayoría compra una sola vez (68,3% one-time).
- **Hallazgo:** pero el 31,7% recurrente genera el 69% del revenue (ticket ~5× mayor).
- **Traducción:** retener un punto de ese núcleo vale más que captar varios one-time; además vuelven rápido (mediana 1,8 días) y a lo mismo (85,5%).
- **Acción:** nudge de recompra a 24–72 h en la misma categoría, dirigido al recurrente (y al one-time de alto ticket para empujarlo al núcleo).

**Prueba de los 30 segundos:** un tercio del ancho = dos tercios del dinero; la caja dice cuándo y cómo actuar.

### ⏳ Tareas pendientes para el equipo (Fase II)

Los héroes quedan como **andamiaje a refinar** (no versión final). Siguiendo el taller (Data-to-Ink, preatentivos, jerarquía, Gestalt, anotaciones, acto de habla):

**1. Pulir los gráficos héroe (estética).** Los borradores están en matplotlib. Sugerencias: migrar a **Plotly** (`go.Figure`) para alimentar el dashboard (Fase III); tipografía y paleta institucional coherentes; afinar la posición de anotaciones, etiquetas y cajas de acción.

**2. Completar el set multi-gráfico (D5 / D2).** Estado por ancla:

| Ancla | Gráfico candidato | Estado |
|---|---|---|
| **PN1 · Palanca A (conversión)** | Revenue en juego en electronics (4.8a) | Borrador |
| **PN2 · Palanca B (retención)** | Concentración de valor recurrentes (4.6) + timing recompra (4.8b) | Borrador |
| **PN3 · Cuándo/con qué** | Intensidad horaria: la mañana convierte ~2× (4.3) | Pendiente |

> Cada gráfica final debe pasar la prueba de 30 s y seguir Contexto → Hallazgo → Traducción de negocio → Acción.

---

## **Pendientes para el paso 5 (§2.3.1) — pasada 2 + congelar contrato Gold (§13)**

Este EDA ya corre sobre Silver/Gold v1 completas (re-fuente §2.3.1 paso 4). Lo que queda
para cerrar la secuencia:

1. **Pasada 2 sobre la Gold** (§2.3.1 paso 5): variables para ML informadas por este EDA,
   tablas/métricas agregadas para el tablero (Kelly), y revisión de
   **correlación/multicolinealidad** (heatmap en §5 como insumo).
2. **Limpieza más allá de dups/nulos** (nota de Yeison): auditar **valores únicos de las
   categóricas/strings** y coherencia. En particular, en Silver los nulos de `brand` (~14%) y
   `macro_category` (~32%) se imputaron al placeholder **`"Unknown"`**; decidir su tratamiento
   definitivo (mantener como categoría, re-derivar desde `category_id`, o marcar como faltante).
   Pendiente también (sin umbral acordado, doc 00 §13): **outliers de precio** y **sesiones-bot**.
3. **Decisión de Sara — flag `sin_navegacion_previa`** (§13): mantener con flag / excluir /
   segmento aparte. Insumo en §5 (tamaño y tasa de etiqueta del flag).
4. **Congelar el contrato del esquema Gold COMPLETO** (§13) tras la pasada 2 + aplicar
   **particionamiento** (doc 02 §3: por fecha/categoría + `ZORDER`) y **exportar la Gold agregada**
   pequeña para Power BI.
5. **Limpieza del Delta temporal** `_tmp_eda_units` al cerrar (no es contrato):
   `dbutils.fs.rm(TMP_UNITS, recurse=True)`.
6. **Alinear con Kelly** (doc 00 §11, §2.3.1 paso 4): este EDA es el insumo de diseño del tablero
   — las dos palancas (electrónica / recurrentes) y el mapa pregunta→gráfico (doc 07 §5).
7. **Coordinar actualización de doc 00 §6** (archivo compartido): los números full-data
   (abandono 51.7%, conv 2.23% por unidad) difieren de los allí citados.
8. **Revisar el split train/test** (decisión ABIERTA, con Sara): el corte Oct/Nov no se ha
   validado; usar §6.1 (shift por Black Friday) como evidencia para decidir el corte temporal
   definitivo. Mantener split **temporal** (nunca aleatorio, por la fuga).
9. **EDA = v1 sobre Silver/Gold actuales.** Si la pasada 2 limpia bots/outliers/`"Unknown"`,
   re-correr el notebook (`FORCE_REBUILD_UNITS=True` si cambia Silver) y refrescar los números de
   la narrativa. Más sensibles: distribuciones por sesión/feature (bots), categoría/marca
   (`"Unknown"`), precio (outliers); el funnel/conversión headline debería ser robusto.